# 06a Run tables, core batches: shadow prices, delivery and credit recovery for smr100, step4, itcfb, itcfbm

**Purpose.** Every batch of ReEDS runs that NREL returned was put through a *check notebook*: a
registry of PASS/FAIL gates (file health, input fingerprints, mandate mechanics, cross-case
consistency) plus the derived per-run tables that the analysis notebooks then read. This notebook
replaces the four check notebooks of the core batches (`step3_checks` for the 25 `smr100` runs,
`step4_checks` for the 120 market-sensitivity and large-reactor runs, `itcfb_checks` for the 19
credit-feedback runs, `itcfbm_checks` for the 24 below-headline credit runs). It computes the same
derived tables from the parquet extracts of notebook 05 and re-runs the gates that can be run from
the extracts. The most-read table is `duals_by_year.csv`: the shadow price of the nuclear capacity
mandate (the marginal system cost of one more MW of required nuclear capacity, which the paper reads
as the required subsidy) for every run and model year, beside the target and the built capacity.

**Inputs.**
- `results/extracts/{smr100,step4,itcfb,itcfbm}/*.parquet` and `results/extracts/key_inventory.parquet`
  (notebook 05_extract_reeds_outputs).
- Casefiles `cases_nuclearlearning_{smr100,step4,itcfb,itcfbm}.csv` at the repository root.
- ReEDS input tables of this fork under `inputs/`: `nuclear_learning/nuclear_cap_trajectory_*.csv`
  (target trajectories), `nuclear_learning/nuclear_cap_mandate_techs_*.csv`,
  `plant_characteristics/*.csv` (drawn plant costs), `financials/*` (deflator, system and technology
  financing, depreciation, construction schedules, regional cost factors, incentives files),
  `load/demand_AEO_2026_baseline.csv`.

**Outputs.** `results/exports/06a/<check notebook>/<file>`, with the check notebook's own folder and
file names so that later ports map one to one:
- `step3_checks/`: `duals_by_year`, `overbuild_by_case`, `eq_flip_comparison`, `checks_summary`,
  `fingerprint_errors`, `fin_mult_noITC_comparison`, `load_invariance`.
- `step4_checks/`: `duals_by_year`, `overbuild_by_case`, `checks_summary`, `pointer_copy_comparison`,
  `switch_echo`, `fingerprint_errors`, `fin_mult_noITC_comparison`, `load_invariance`.
- `itcfb_checks/`: `duals_by_year`, `trajectory_gaps`, `rate_recovery`, `fbK_calibration`,
  `fbA_collapse`, `incentives_echo`, `checks_summary`, `fingerprint_errors`, `base_identity`,
  `load_invariance`.
- `itcfbm_checks/`: `trajectory_gaps`, `delivery_outcomes`, `incentives_echo`, `checks_summary`,
  `rate_recovery`, `sibling_rate_offset`, `fingerprint_errors`, `base_identity`, `load_invariance`.

**Paper items.** None directly. The tables feed notebooks 07 (Table 1, Fig 2a, SF8 to SF10, ST2,
ST11, ST12, ST15), 08 (SF13, SF23, ST14), 09 (SF11, SF12, ST3), 10 (Fig 3, Fig 4, Table 2), 11 (Fig 5),
13a/13b (SF24, ST16) and 14 (SF16, SF17, SF25, ST5).

**Runtime.** About 1 minute on this machine (extracts only; no h5 file is opened).

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import nlpaper
from nlpaper import paths, plotstyle as ps, h5io
nlpaper.setup(threads=4)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


{'python': '3.12.3',
 'numpy': '1.26.4',
 'pandas': '2.2.2',
 'threads': 4,
 'threadpoolctl': True,
 'env_set': {'OMP_NUM_THREADS': '4',
  'MKL_NUM_THREADS': '4',
  'OPENBLAS_NUM_THREADS': '4',
  'NUMEXPR_NUM_THREADS': '4',
  'NUMBA_NUM_THREADS': '4'},
 'matplotlib': '3.8.4',
 'seed_note': None}

## 1. What the check notebooks did, and what this notebook keeps

A *check* is one PASS/FAIL (or INFO, context only) gate that the source check notebook ran on every
delivered batch before any analysis read it. The gates fall into phases: **A** file health (every
run returned, same report tables, solver residuals small, the sequential solve mode), **B** input
fingerprints (the run used the intended plant costs, financing multipliers and incentives; no
investment tax credit reached nuclear unless the design fed one), **C** mandate mechanics (the
capacity meets the target; the shadow price is positive only where the target binds) or, for the
credit batches, the pre-registered criteria of each arm, **D/E/F** cross-case consistency (exogenous
load identical across runs, no NaN, national totals equal regional sums). Each registry was written
to `checks_summary.csv`. The credit-feedback analyses (notebook 09) gate on it: no FAIL is allowed
in phases A, B and D.

This notebook runs the same gates on the extracts. Three kinds of gate change:

- **Adapted to the key inventory.** File census, report-table (key) sets, the NaN/infinity scan and
  the non-negativity scan of whole files now read `key_inventory.parquet`, which notebook 05 wrote
  from the full h5 files (`returned`, `keys`, `n_nonfinite`, `min_<key>`).
- **Adapted to the incentives files.** The credit batches took their expected fed rates from the
  required-ITC table `t09_required_itc.csv`, which notebook 07 produces from this notebook's
  `duals_by_year`. To avoid the loop, the fed rates here come from the incentives files that the
  runs read (`inputs/financials/incentives_obbba_itcfb*_p50.csv`). The test "the incentives files
  equal the headline rate schedule" and the column `itc_frac_expected` of `incentives_echo.csv` move
  to notebook 09.
- **Dropped.** Archive bookkeeping: the pilot-copy hash test (step3 A2), the zip-extraction
  manifest test (step4 A2), the folder census counts of the old delivery folder; and the two
  tests that opened HDF5 *input* files (`inputs/profiles_demand/*.h5`: step3 E5, step4 E3). The
  archive manifest `data/reeds_outputs_manifest.csv` holds the SHA-256 of every run file.

Each registry row carries a `port` column: `ported` (same test, same numbers), `adapted` (same
purpose, now on the key inventory or the incentives files) or `moved` (a note that points to the
notebook that now runs the test).

### Shared helpers (candidates for `nlpaper`)

The next cell holds the machinery that the four check notebooks each carried a copy of: the
extract reader keyed by case, the registry, the target trajectories, the shadow-price table
(`duals_by_year`), the binding-year summary (`overbuild_by_case`), the delivery tiers, the
financing-multiplier replication and the recovery of the applied credit rate from the two
financing multipliers, and the generic sanity gates. Notebook 06b (the rate-design batches) needs
the same functions; they are kept in this one marked cell so the lead can move them into `nlpaper`
in one step.

**Shadow-price convention.** ReEDS reports the mandate's shadow price twice. `nuclear_cap_price_raw`
is the marginal of the mandate constraint in objective units. `nuclear_cap_price` is that marginal
divided by `cost_scale` (1 in these runs) and by `pvf_onm(t)`, the present-value factor of operating
costs (the inverse of the capital recovery factor, 14.7645 in every solve year from 2026 in the
sequential solve). The result is an annual rental price in 2004 dollars per MW-year. Dividing by 1000
gives $/kW-yr; multiplying by `TO2024 = 1/Deflator(2024)` gives 2024 dollars (`dual_2024_MWyr`).
Later notebooks multiply the rental price by `pvf_onm` to get the up-front required support per MW.
A run whose file has no `nuclear_cap_price` table never had a binding mandate: ReEDS drops an
all-zero table, so its shadow price is zero in every year.

**Binding year.** A mandated year (target > 0) is *binding* when the shadow price exceeds
$1/MW-yr; the capacity then equals the target within 5 MW. A mandated year is *slack* when the
capacity exceeds the target by more than 100 MW; the shadow price is then zero.

In [2]:
# ==================================================================================
# SHARED HELPER (06a and 06b): extract reader by case, registry, trajectories,
# shadow-price table, binding years, delivery tiers, financing machinery, sanity gates.
# ==================================================================================
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

INPUTS = paths.reeds_inputs()
FIN_DIR = INPUTS / "financials"
PC_DIR = INPUTS / "plant_characteristics"
NL_DIR = INPUTS / "nuclear_learning"
OUT = paths.exports("06a")

def out_dir(name):
    p = OUT / name
    p.mkdir(parents=True, exist_ok=True)
    return p

# ---- dollars: 2022$ -> 2004$ (ReEDS-internal) and 2004$ -> 2024$ --------------------
DEFL = pd.read_csv(paths.deflator_path())
DEFL.columns = ["t", "Deflator"]
DEFL = DEFL.set_index("t")["Deflator"]
D2022 = float(DEFL.loc[2022])
TO2024 = 1.0 / float(DEFL.loc[2024])

NUC = ["nuclear", "nuclear-smr"]            # the two nuclear techs; nothing else contains "nuclear"
DUAL_KEYS = {"nuclear_cap_price", "nuclear_cap_price_raw",
             "nuclear_cap_price_ub", "nuclear_cap_price_ub_raw"}
NONNEG = ["cap", "cap_nat", "load_rt", "stor_in", "stor_out", "curt_ann", "hours"]

# ---- runs: case -> batch, and the report-table (key) set of each run ----------------
INV = pd.read_parquet(paths.require(paths.EXTRACTS / "key_inventory.parquet",
                                    "notebook 05_extract_reeds_outputs"))
RUN_BATCH, KEYS = {}, {}
for r in INV.itertuples():
    if r.batch not in ("smr100", "step4", "itcfb", "itcfbm"):
        continue
    assert r.case not in RUN_BATCH, f"case name {r.case} appears in two core batches"
    RUN_BATCH[r.case] = r.batch
    KEYS[r.case] = set(r.keys.split(";")) if r.returned else set()

_SPLIT = {}

def load(case, key):
    # One report table of one run, as the h5 reader returned it (columns in h5 order, no `case`).
    # Mixed extracts (cap, cap_new_ann, gen_ann, ret_ann, systemcost_techba) hold nuclear rows by
    # region and every other tech summed nationally (r = "national"). Do not modify the result.
    b = RUN_BATCH[case]
    if (b, key) not in _SPLIT:
        df = h5io.load(b, key)
        _SPLIT[(b, key)] = {c: g.drop(columns="case").reset_index(drop=True)
                            for c, g in df.groupby("case", sort=False)}
    d = _SPLIT[(b, key)]
    if case not in d:
        raise KeyError(f"{key} not in {b}/{case}")
    return d[case]

def h5_keys(case):
    return sorted(KEYS[case])

def nuc_slice(case, key):
    df = load(case, key)
    df = df[df["i"].isin(NUC)]
    dims = [c for c in df.columns if c != "Value"]
    return df.set_index(dims)["Value"]

def switch(casefile, case, name):
    # One switch value of one case; an empty cell falls back to the Default Value column.
    v = casefile.loc[name, case]
    if pd.isna(v) or str(v).strip() == "":
        v = casefile.loc[name, "Default Value"]
    return "" if pd.isna(v) else str(v).strip()

# ---- the check registry ---------------------------------------------------------------
REGISTRY = {}

def registry(name):
    rows = REGISTRY.setdefault(name, [])
    def record(phase, check, status, detail="", port="ported"):
        assert status in ("PASS", "FAIL", "BLOCKED", "INFO")
        rows.append(dict(phase=phase, check=check, status=status, detail=str(detail), port=port))
        print(f"[{status}] {phase} :: {check}" + (f" -- {detail}" if detail else ""))
    return record

def write_registry(name):
    s = pd.DataFrame(REGISTRY[name])
    s.to_csv(out_dir(name) / "checks_summary.csv", index=False)
    print(name, s["status"].value_counts().to_dict())
    return s

# ---- target trajectories and input tables -----------------------------------------------
def trajectory(scen):
    tr = pd.read_csv(NL_DIR / f"nuclear_cap_trajectory_{scen}.csv")
    tr.columns = [c.lstrip("*") for c in tr.columns]
    return tr.set_index(tr.columns[0])["MW"]

def plantchar(name):
    df = pd.read_csv(PC_DIR / f"{name}.csv")
    df.columns = [c.lstrip("*").lower() for c in df.columns]
    return df.set_index("t")

# ---- shadow prices, duals_by_year, overbuild_by_case -------------------------------------
def dual_series(case):
    # (converted, raw) shadow price by year; empty when the run has no binding mandate.
    if "nuclear_cap_price" in KEYS[case]:
        return (load(case, "nuclear_cap_price").set_index("t")["Value"],
                load(case, "nuclear_cap_price_raw").set_index("t")["Value"])
    return pd.Series(dtype=float), pd.Series(dtype=float)

def duals_table(cases, years, traj, mand_cap, dual, raw):
    # One row per (case, model year): target, built capacity of the mandated tech, slack, shadow
    # price (2004$/MW-yr, raw objective units, 2024$/MW-yr). Also returns the three mandate gates
    # evaluated on the unrounded capacity: floor violations, binding years off the target,
    # positive prices in slack years.
    rows, viol, bind_bad, slack_bad = [], [], [], []
    for c in cases:
        tr, s, d = traj[c], mand_cap[c], dual[c]
        mand_years = [t for t in years if tr is not None and tr.get(t, 0) > 0]
        for t in years:
            req = float(tr.get(t, 0.0)) if tr is not None else 0.0
            have = float(s.get(t, 0.0))
            dv = float(d.get(t, 0.0))
            rows.append(dict(case=c, t=t, mandate_MW=req, cap_MW=round(have, 1),
                             slack_MW=round(have - req, 1), dual_2004_MWyr=dv,
                             dual_raw=float(raw[c].get(t, 0.0)),
                             dual_2024_MWyr=dv * TO2024))
            if t in mand_years:
                if have - req < -5:
                    viol.append((c, t, round(have - req, 1)))
                if dv > 1.0 and abs(have - req) > 5:
                    bind_bad.append((c, t, dv, round(have - req, 1)))
                if have - req > 100 and dv > 1e-3:
                    slack_bad.append((c, t, dv, round(have - req, 1)))
    return pd.DataFrame(rows), viol, bind_bad, slack_bad

def overbuild_table(duals_tbl, cases, years, traj):
    # Binding and slack years of each run (see the definitions above).
    ob_rows = []
    for c in cases:
        tr = traj[c]
        mand_years = [t for t in years if tr.get(t, 0) > 0]
        sub = duals_tbl[(duals_tbl.case == c) & duals_tbl.t.isin(mand_years)]
        slackers = sub[sub.slack_MW > 100]
        first = (int(sub[sub.dual_2004_MWyr > 1.0].t.min())
                 if (sub.dual_2004_MWyr > 1.0).any() else None)
        ob_rows.append(dict(case=c, n_mandated_years=len(mand_years),
                            n_binding_years=int((sub.dual_2004_MWyr > 1.0).sum()),
                            n_slack_years=len(slackers), first_binding_year=first,
                            max_overbuild_ratio=round(float((sub.cap_MW / sub.mandate_MW).max()), 2)))
    return pd.DataFrame(ob_rows)

def gap_tier(gaps, case):
    # Delivery tier of one credit run against its target trajectory (itcfb convention):
    # exact | reproduces | partial | no.
    sub = gaps[gaps["case"] == case]
    shortfall = (-sub["gap_MW"]).clip(lower=0.0)
    if float(shortfall.max()) <= 5.0 and float(sub["gap_MW"].abs().max()) <= 5.0:
        return "exact"
    gate = sub[(sub["status"] == "rate") | (sub["t"] == 2050)]
    g_short = (-gate["gap_MW"]).clip(lower=0.0)
    tol1 = np.maximum(0.05 * gate["mandate_MW"], 500.0)
    if bool((g_short <= tol1).all()):
        return "reproduces"
    end = gate[gate["t"] == 2050]
    if len(end) and float(-end["gap_MW"].iloc[0]) <= 0.15 * float(end["mandate_MW"].iloc[0]):
        return "partial"
    return "no"

# ---- financing machinery (inputs/financials; mirrors 2_financials.gms) -------------------
YEARS_FIN = np.arange(2010, 2051)

def yi(t):
    return int(t - YEARS_FIN[0])

_sys_fin = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv")
_sys_fin = _sys_fin.merge(pd.read_csv(FIN_DIR / "inflation_default.csv"), on="t", how="left")
_sys_fin["d_nom"] = ((1 - _sys_fin["debt_fraction"]) * (_sys_fin["rroe_nom"] - 1)
                     + _sys_fin["debt_fraction"] * (_sys_fin["interest_rate_nom"] - 1)
                       * (1 - _sys_fin["tax_rate"]) + 1)
_sys_fin["d_real"] = _sys_fin["d_nom"] / _sys_fin["inflation_rate"]

def _on_years(col):
    s = _sys_fin.set_index("t")[col].reindex(range(1990, YEARS_FIN[-1] + 1)).ffill()
    return s.loc[YEARS_FIN].to_numpy(float)

IB, TAX, D_NOM, D_REAL = (_on_years(c) for c in
                          ["interest_rate_nom", "tax_rate", "d_nom", "d_real"])
DEP = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
CS_MC = pd.read_csv(FIN_DIR / "construction_schedules_mc.csv")
CC_EXPS = np.append([0.0], np.arange(0.5, 10.5, 1.0))   # header 'NA' row -> 0; rows t=0..9 -> 0.5..9.5
assert len(CS_MC) == len(CC_EXPS), (len(CS_MC), len(CC_EXPS))

def ccmult_reeds(sched_name, ib):
    x = pd.to_numeric(CS_MC[str(sched_name)], errors="coerce").fillna(0.0).to_numpy()
    return 1.0 + float(np.sum(x * (ib ** CC_EXPS - 1.0)))

def _fin_tech(fin_suffix):
    ft = pd.read_csv(FIN_DIR / f"financials_tech_{fin_suffix}.csv")
    ft.columns = [c.lstrip("*") for c in ft.columns]
    return ft[ft["i"].isin(["Nuclear", "Nuclear-SMR"])]

def national_base(fin_suffix):
    # ccmult(i,t) x 1/(1-tax) x (1-tax*pv_dep(i,t)) x risk_mult(i,t) x eval_adj: the region-free
    # part of cost_cap_fin_mult_noITC (degradation_adj = 1 for nuclear).
    out = {}
    for iname, grp in _fin_tech(fin_suffix).groupby("i"):
        grp = grp.set_index("t").reindex(YEARS_FIN).ffill().bfill()
        dep_frac = DEP[str(int(grp["depreciation_sch"].iloc[0]))].to_numpy(float)
        pv_dep = np.array([np.sum(dep_frac / dn ** np.arange(1, len(dep_frac) + 1)) for dn in D_NOM])
        eval_p = float(grp["eval_period"].iloc[0])
        risk = 1.0 + grp["finance_diff_real"].to_numpy(float) * (
            (1 - (1 / D_REAL) ** eval_p) / (D_REAL - 1.0))
        sys_pvf = (1 - (1 / D_REAL) ** (30 - 1)) / (D_REAL - 1.0) + 1
        tech_pvf = (1 - (1 / D_REAL) ** (eval_p - 1)) / (D_REAL - 1.0) + 1
        ccm = np.array([ccmult_reeds(grp.loc[t, "construction_sch"], IB[yi(t)]) for t in YEARS_FIN])
        out[iname.lower()] = pd.Series(ccm / (1 - TAX) * (1 - TAX * pv_dep) * risk * (sys_pvf / tech_pvf),
                                       index=YEARS_FIN)
    return out

def fin_mult_decomposition(cases, fin_of):
    # h5(i,r,t) = base(i,t) x regional factor(i,r): returns the comparison table and the factors.
    fin_rows, reg_factors = [], {}
    for c in cases:
        base = national_base(fin_of[c])
        h5f = load(c, "cost_cap_fin_mult_noITC")
        h5f = h5f[h5f["i"].isin(NUC)]
        for iname, grp in h5f.groupby("i"):
            piv = grp.pivot_table(index="r", columns="t", values="Value")
            exp = base[iname].reindex(piv.columns)
            ratio = piv / exp.values[None, :]
            reg = ratio.mean(axis=1)
            reg_factors[(c, iname)] = reg
            resid = (piv - reg.values[:, None] * exp.values[None, :]).abs()
            fin_rows.append(dict(
                case=c, i=iname, n_regions=len(piv),
                ratio_spread_over_t=float((ratio.max(axis=1) - ratio.min(axis=1)).max()),
                worst_resid_after_regional=float(resid.max().max()),
                reg_factor_mean=float(reg.mean()), reg_factor_min=float(reg.min()),
                reg_factor_max=float(reg.max())))
    return pd.DataFrame(fin_rows), reg_factors

def pv_dep_nuclear(fin_suffix):
    # Present value of depreciation by year for each nuclear tech.
    out = {}
    for iname, grp in _fin_tech(fin_suffix).groupby("i"):
        grp = grp.set_index("t").reindex(YEARS_FIN).ffill().bfill()
        dep_frac = DEP[str(int(grp["depreciation_sch"].iloc[0]))].to_numpy(float)
        out[iname.lower()] = pd.Series(
            [np.sum(dep_frac / dn ** np.arange(1, len(dep_frac) + 1)) for dn in D_NOM], index=YEARS_FIN)
    return out

def recovered_m(case, fin_suffix):
    # Monetized ITC rate per (i, r, t) from the ratio of the two financing multipliers
    # (reeds/financials.py:683-694): m = (1 - ratio)(1 - tax*PVdep) / (1 - tax*PVdep/2).
    fm = load(case, "cost_cap_fin_mult")
    fno = load(case, "cost_cap_fin_mult_noITC")
    fm = fm[fm["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
    fno = fno[fno["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
    j = pd.concat([fm, fno], axis=1, keys=["fm", "fno"]).dropna().reset_index()
    pv = pv_dep_nuclear(fin_suffix)
    pd_ = np.array([pv[i].loc[t] for i, t in zip(j["i"], j["t"])])
    tau = TAX[[yi(t) for t in j["t"]]]
    ratio = j["fm"].to_numpy() / j["fno"].to_numpy()
    j["m_rec"] = (1.0 - ratio) * (1.0 - tau * pd_) / (1.0 - tau * pd_ / 2.0)
    return j

def cost_fingerprint(cases, pc_of, learning_of=None):
    # Capital cost in the run (cost_cap, 2004$/MW) against the plant file (2022$/kW) x 1000 x D2022.
    rows = []
    for c in cases:
        cc = load(c, "cost_cap")
        cc = cc[cc["i"].isin(NUC)].set_index(["i", "t"])["Value"]
        for itech in NUC:
            for t, occ_kw in plantchar(pc_of[c][itech])["capcost"].items():
                if (itech, t) not in cc.index:
                    continue
                expect = occ_kw * 1000.0 * D2022
                got = float(cc.loc[(itech, t)])
                row = dict(case=c)
                if learning_of is not None:
                    row["learning"] = learning_of[c]
                row.update(i=itech, t=t, expect=expect, got=got, rel_err=abs(got - expect) / expect)
                rows.append(row)
    return pd.DataFrame(rows)

# ---- generic sanity gates ------------------------------------------------------------------
def gate_census(record, phase, batch, cases, check):
    inv = INV[INV.batch == batch].set_index("case")
    missing = [c for c in cases if c not in inv.index or not inv.loc[c, "returned"]]
    record(phase, check, "PASS" if not missing else "FAIL",
           f"missing: {missing}" if missing else
           f"{len(cases)} runs returned (key_inventory: batch {batch})", port="adapted")
    return missing

def gate_keys(record, phase, cases, check, ref=None, ref_check=None):
    keysets = {c: KEYS[c] for c in cases}
    union = set.union(*keysets.values())
    bad = {}
    for c in cases:
        diff = (union - keysets[c]) | (keysets[c] - union)
        if diff - DUAL_KEYS:
            bad[c] = sorted(diff - DUAL_KEYS)
    absent = {c: sorted(DUAL_KEYS - keysets[c]) for c in cases if DUAL_KEYS - keysets[c]}
    record(phase, check, "PASS" if not bad else "FAIL",
           f"key counts {sorted(set(len(keysets[c]) for c in cases))}"
           + (f"; unexpected differences: {bad}" if bad else
              f"; runs without a shadow-price table: {sum('nuclear_cap_price' in v for v in absent.values())}"),
           port="adapted")
    if ref is not None:
        ref_keys = KEYS[ref]
        drift = sorted(((union - ref_keys) | (ref_keys - union)) - DUAL_KEYS)
        record(phase, ref_check, "PASS" if not drift else "FAIL",
               f"drift vs {ref}: {drift}" if drift else f"{len(ref_keys)} keys, no drift",
               port="adapted")
    return keysets, union

def gate_solve_health(record, phase, cases, expect_years, zrep_years=None):
    worst_z, worst_at = 0.0, None
    for c in cases:
        ec = load(c, "error_check").set_index("*")["Value"]
        m = float(ec.abs().max())
        if m > worst_z:
            worst_z, worst_at = m, c
    record(phase, "the solver residuals are small in every file",
           "PASS" if worst_z < 1e-2 else "FAIL", f"worst |residual| {worst_z:.2e} at {worst_at}")
    obj = {c: float(load(c, "objfn_raw")["Value"].iloc[0]) for c in cases}
    record(phase, "the objective value is a normal positive number in every file",
           "PASS" if all(np.isfinite(v) and v > 0 for v in obj.values()) else "FAIL",
           f"range {min(obj.values()):.4e} to {max(obj.values()):.4e}")
    bad_years = {c: y for c in cases if (y := sorted(load(c, "cap")["t"].unique())) != expect_years}
    record(phase, "the model years are correct in every file (annual 2031-2035 block present)",
           "PASS" if not bad_years else "FAIL",
           f"expected {expect_years}" + (f"; wrong in {bad_years}" if bad_years else ""))
    # pvf_onm = 1/crf(t): flat from 2026 on; pvf_capital = 1 and cost_scale = 1 in sequential solves.
    worst = dict(pvfc=0.0, pvfo=0.0, cs=0.0, zrep=0)
    for c in cases:
        pvfc = load(c, "pvf_capital").set_index("t")["Value"]
        pvfo = load(c, "pvf_onm").set_index("t")["Value"]
        pvfo = pvfo[pvfo.index >= 2026]
        cs = load(c, "cost_scale")["Value"].iloc[0]
        worst["pvfc"] = max(worst["pvfc"], float((pvfc - 1.0).abs().max()))
        worst["pvfo"] = max(worst["pvfo"], float((pvfo - pvfo.mean()).abs().max()))
        worst["cs"] = max(worst["cs"], abs(float(cs) - 1.0))
        worst["zrep"] = max(worst["zrep"], abs(len(load(c, "z_rep")) - len(expect_years)))
    ok = worst["pvfc"] < 1e-9 and worst["pvfo"] < 1e-3 and worst["cs"] < 1e-9 and worst["zrep"] == 0
    pvf26 = float(load(cases[0], "pvf_onm").set_index("t")["Value"].loc[2026])
    record(phase, "every run used the sequential solve mode", "PASS" if ok else "FAIL",
           f"pvf_capital==1 (max dev {worst['pvfc']:.1e}); pvf_onm flat from 2026 at {pvf26:.4f} "
           f"(max dev {worst['pvfo']:.1e}); cost_scale==1; z_rep has {len(expect_years)} years in every file")
    return obj

def gate_data_scan(record, phase, cases):
    # NaN / infinity in any table and negative values in never-negative tables: from the key
    # inventory, which notebook 05 computed on every Value of every table of the full h5 file.
    inv = INV.set_index(["batch", "case"])
    nan_bad, neg_bad, nuc_neg = [], [], []
    for c in cases:
        row = inv.loc[(RUN_BATCH[c], c)]
        if row["n_nonfinite"] > 0:
            nan_bad.append((c, row["nonfinite_keys"], int(row["n_nonfinite"])))
        for k in NONNEG:
            mn = row[f"min_{k}"]
            if pd.notna(mn) and float(mn) < -1.0:
                neg_bad.append((c, k, float(mn)))
        for k in ["cap_new_ann", "gen_ann"]:
            df = load(c, k)
            nuc = df[df["i"].isin(NUC)]
            if len(nuc) and float(nuc["Value"].min()) < -1.0:
                nuc_neg.append((c, k, float(nuc["Value"].min())))
    n_keys = max(len(KEYS[c]) for c in cases)
    record(phase, "no data value is NaN or infinite in any file", "PASS" if not nan_bad else "FAIL",
           nan_bad[:5] or f"all {n_keys} keys x {len(cases)} files scanned (key_inventory.n_nonfinite)",
           port="adapted")
    record(phase, "no negative values occur where values must not be negative",
           "PASS" if not neg_bad else "FAIL", neg_bad[:5] or f"checked {NONNEG} (key_inventory.min_*)",
           port="adapted")
    record(phase, "the nuclear build and generation data is not negative",
           "PASS" if not nuc_neg else "FAIL",
           nuc_neg[:5] or f"cap_new_ann and gen_ann nuclear rows >= 0, all {len(cases)} cases "
           "(storage charging and upgrades make some non-nuclear rows negative; a normal convention)")

def gate_national_totals(record, phase, cases):
    tot_bad = []
    for c in cases:
        for key, nat_key in [("cap", "cap_nat"), ("gen_ann", "gen_ann_nat"),
                             ("cap_new_ann", "cap_new_ann_nat"), ("ret_ann", "ret_ann_nat")]:
            s = load(c, key).groupby(["i", "t"])["Value"].sum()
            nat_df = load(c, nat_key)
            n = nat_df.groupby([col for col in nat_df.columns if col in ("i", "t")])["Value"].sum()
            j = pd.concat([s, n], axis=1, keys=["sum", "nat"]).fillna(0.0)
            rel = float(((j["sum"] - j["nat"]).abs() / j["nat"].abs().clip(lower=1.0)).max())
            if rel > 1e-3:
                tot_bad.append((c, key, rel))
    record(phase, "the national totals equal the sum of the regional data",
           "PASS" if not tot_bad else "FAIL",
           tot_bad[:5] or f"cap, gen_ann, cap_new_ann, ret_ann; all {len(cases)} cases within 1e-3")

def gate_dual_conversion(record, phase, cases, dual, raw):
    # report.gms: converted = raw / (cost_scale x pvf_onm(t)).
    worst = 0.0
    for c in cases:
        if not len(dual[c]):
            continue
        pvf = load(c, "pvf_onm").set_index("t")["Value"]
        cs = float(load(c, "cost_scale")["Value"].iloc[0])
        implied = raw[c] / dual[c] / cs
        rel = (implied - pvf.reindex(implied.index)).abs() / pvf.reindex(implied.index)
        worst = max(worst, float(rel.max()))
    record(phase, "the dual conversion is correct (raw = cost_scale x pvf_onm x converted)",
           "PASS" if worst < 1e-4 else "FAIL", f"max rel err {worst:.2e} across all cases with duals")

def gate_dropped_load(record, phase, cases):
    dl_bad, hist = [], {}
    for c in cases:
        dl = load(c, "dropped_load")
        modern = float(dl.loc[dl["t"] >= 2026, "Value"].sum())
        hist[c] = float(dl.loc[dl["t"] < 2026, "Value"].sum())
        if modern >= 1.0:
            dl_bad.append((c, modern))
    spread = max(hist.values()) - min(hist.values())
    record(phase, "no load is dropped after 2025", "PASS" if not dl_bad else "FAIL",
           dl_bad[:5] or f"all {len(cases)} cases clean; historical 2010-2023 artifact "
           f"{np.mean(list(hist.values()))/1e6:.2f} TWh, case spread {spread:.1f} MWh")

def load_invariance(cases, ref_of):
    # Largest relative difference of the busbar load by (region, year) against a reference run.
    rows = []
    for c in cases:
        b = ref_of[c]
        a = load(c, "load_rt").set_index(["r", "t"])["Value"]
        r = load(b, "load_rt").set_index(["r", "t"])["Value"]
        j = pd.concat([a, r], axis=1, keys=["c", "ref"]).fillna(0.0)
        rows.append(dict(case=c, vs=b,
                         max_rel_diff=float(((j["c"] - j["ref"]).abs() / j["ref"].abs().clip(lower=1.0)).max())))
    return pd.DataFrame(rows)

def input_identity(cases, base_of, keys=("cost_cap", "cost_cap_fin_mult_noITC")):
    # Nuclear rows of two runs that point at the same nuclear input files must agree exactly.
    rows = []
    for c in cases:
        b = base_of[c]
        for key in keys:
            a, r = nuc_slice(c, key), nuc_slice(b, key)
            j = pd.concat([a, r], axis=1, keys=["x", "base"]).fillna(0.0)
            d = (j["x"] - j["base"]).abs()
            rows.append(dict(case=c, base=b, key=key, n_points=len(j), max_abs_diff=float(d.max()),
                             max_rel_diff=float((d / j["base"].abs().clip(lower=1e-9)).max())))
    return pd.DataFrame(rows)

def itc_wedge_leak(pairs):
    # Non-nuclear ITC wedge (fin_mult_noITC - fin_mult) of a credit run vs its base run.
    bad = []
    for c, b in pairs:
        w = {}
        for name, run in [("fb", c), ("base", b)]:
            fm = load(run, "cost_cap_fin_mult")
            fno = load(run, "cost_cap_fin_mult_noITC")
            fm = fm[~fm["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
            fno = fno[~fno["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
            w[name] = fno - fm
        j = pd.concat([w["fb"], w["base"]], axis=1, keys=["fb", "base"]).dropna()
        d = float((j["fb"] - j["base"]).abs().max())
        if d > 1e-6:
            bad.append((c, b, d))
    return bad

print(f"core runs: {pd.Series(RUN_BATCH).value_counts().to_dict()}")
print(f"2022$->2004$ deflator: {D2022:.6f}; 2004$->2024$: {TO2024:.4f}")

core runs: {'step4': 120, 'smr100': 28, 'itcfbm': 24, 'itcfb': 19}
2022$->2004$ deflator: 0.645694; 2004$->2024$: 1.6590


## 2. The smr100 batch (`step3_checks`): the 25 production runs

The `smr100` casefile holds 18 SMR-mandate runs (six deployment schedules, each at the p05, p50 and
p95 cost draw), six large-reactor runs (`large100_<schedule>_p50`) and one equality-mandate twin
(`smr100_eia_p50_eq`: floor and ceiling, to show that the constraint form changes nothing). The
extracts also hold three pilot runs (`smr100_eia_{lo,mid,hi}`, an earlier vintage with a shorter year
set), which only test E6 reads. Nothing below includes them otherwise.

The derived tables and their readers:
- `duals_by_year.csv`: shadow price, target and built capacity for every run and model year. Read by
  notebook 07 (required support, Table 1, t08/t09), 08, 09, 10, 11, 13a/13b and 14. It is the
  canonical base-run shadow-price table of the paper.
- `overbuild_by_case.csv`: binding and slack years per run. Read by 07 and 08.
- `eq_flip_comparison.csv`: national capacity and generation of the equality twin vs the floor run.
  Read by 07 (the note that the equality form leaves the results unchanged).

In [3]:
record = registry("step3_checks")
S3 = out_dir("step3_checks")

cases_csv = pd.read_csv(paths.casefile("smr100"), index_col=0)
CASES = [c for c in cases_csv.columns if c != "Default Value"]
assert len(CASES) == 25, f"the smr100 casefile has 25 production runs, got {len(CASES)}"
sw = lambda c, n: switch(cases_csv, c, n)

META = {}
for c in CASES:
    ts = sw(c, "GSw_NuclearCapMandateTechScen")
    META[c] = dict(scen=sw(c, "GSw_NuclearCapMandateScen"),
                   mode=int(float(sw(c, "GSw_NuclearCapMandate"))), techscen=ts,
                   mandated_tech={"smr": "nuclear-smr", "large": "nuclear"}[ts],
                   pc={"nuclear": sw(c, "plantchar_nuclear"), "nuclear-smr": sw(c, "plantchar_nuclear_smr")},
                   fin=sw(c, "financials_tech_suffix"))
SMR_CASES = [c for c in CASES if META[c]["techscen"] == "smr"]
LARGE_CASES = [c for c in CASES if META[c]["techscen"] == "large"]
SCHEDULES = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
REF = "smr100_eia_p50"
PILOTS = sorted(c for c in RUN_BATCH if RUN_BATCH[c] == "smr100" and c not in CASES)
print(f"{len(SMR_CASES)} smr100 + {len(LARGE_CASES)} large100 runs; pilots in the extracts: {PILOTS}")

# ---- phase A: file health -----------------------------------------------------------------
gate_census(record, "A", "smr100", CASES, "one output file exists for each of the 25 cases")
record("A", "extra files in the folder are identified", "INFO",
       f"{len(PILOTS)} pilot runs in the smr100 extracts (key_inventory.pilot): {PILOTS}", port="adapted")
record("A", "the three extra files are copies of the old pilot files", "INFO",
       "dropped in the port: archive bookkeeping; data/reeds_outputs_manifest.csv holds every file's "
       "SHA-256", port="moved")
keysets, union = gate_keys(record, "A", CASES,
                           "all 25 files have the same data keys (dual keys can be absent when zero)")
EXPECT_YEARS = [int(y) for y in sw(REF, "yearset").split("_")]
YEARS_RUN = sorted(load(REF, "cap")["t"].unique())
obj = gate_solve_health(record, "A", CASES, EXPECT_YEARS)

19 smr100 + 6 large100 runs; pilots in the extracts: ['smr100_eia_hi', 'smr100_eia_lo', 'smr100_eia_mid']
[PASS] A :: one output file exists for each of the 25 cases -- 25 runs returned (key_inventory: batch smr100)
[INFO] A :: extra files in the folder are identified -- 3 pilot runs in the smr100 extracts (key_inventory.pilot): ['smr100_eia_hi', 'smr100_eia_lo', 'smr100_eia_mid']
[INFO] A :: the three extra files are copies of the old pilot files -- dropped in the port: archive bookkeeping; data/reeds_outputs_manifest.csv holds every file's SHA-256
[PASS] A :: all 25 files have the same data keys (dual keys can be absent when zero) -- key counts [209]; runs without a shadow-price table: 0
[PASS] A :: the solver residuals are small in every file -- worst |residual| 7.36e-04 at smr100_eia_p05
[PASS] A :: the objective value is a normal positive number in every file -- range 1.4470e+12 to 1.8856e+12
[PASS] A :: the model years are correct in every file (annual 2031-2035 block present) --

[PASS] A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 at 14.7645 (max dev 9.5e-07); cost_scale==1; z_rep has 16 years in every file


### Phase B: input fingerprints

The runs must have used the intended inputs; NREL did not return the input files, so each gate
recomputes the expected value from this fork's input tables. B1 compares the capital cost with the
plant files (the per-point table is `fingerprint_errors.csv`). B2 to B4 compare heat rate, variable
and fixed O&M. B5 to B9 replicate the financing multiplier `cost_cap_fin_mult_noITC` as a national
base times a regional factor (`fin_mult_noITC_comparison.csv`). B10 to B13 confirm that no
investment tax credit (ITC) reaches nuclear: the paper reads the shadow price as the full required
subsidy, which holds only when no other subsidy reaches nuclear. B14 checks the mandate-technology
input files.

In [4]:
# ---- B1: capital cost fingerprint ---------------------------------------------------------
fp = cost_fingerprint(CASES, {c: META[c]["pc"] for c in CASES})
fp.to_csv(S3 / "fingerprint_errors.csv", index=False)
w = fp.loc[fp.rel_err.idxmax()]
record("B", "the capital cost data agrees with the plant input files (all 25 cases)",
       "PASS" if fp.rel_err.max() < 1e-4 else "FAIL",
       f"{len(fp)} (case,tech,year) points; max rel err {fp.rel_err.max():.2e} at ({w.case}, {w.i}, {w.t})")

# ---- B2-B4: heat rate, VOM, FOM -------------------------------------------------------------
hr_bad = []
for c in CASES:
    hr = load(c, "heat_rate")
    hr = hr[hr["i"].isin(NUC)].groupby("i")["Value"].agg(["min", "max"])
    for itech in hr.index:
        exp = float(plantchar(META[c]["pc"][itech])["heatrate"].iloc[-1])
        if not (abs(hr.loc[itech, "min"] - exp) < 0.05 and abs(hr.loc[itech, "max"] - exp) < 0.6):
            hr_bad.append((c, itech, hr.loc[itech].to_dict(), exp))
record("B", "the heat rate data agrees with the plant input files", "PASS" if not hr_bad else "INFO",
       hr_bad[:3] if hr_bad else "both techs, all cases (old vintages can differ a little)")

vom_bad = []
for c in CASES:
    vom = load(c, "cost_vom")
    for itech in NUC:
        v = vom[vom["i"] == itech]
        if not len(v):
            continue
        got = float(v["Value"].mode().iloc[0])
        exp = float(plantchar(META[c]["pc"][itech])["vom"].iloc[-1]) * D2022
        if abs(got - exp) / exp > 0.02:
            vom_bad.append((c, itech, round(got, 3), round(exp, 3)))
record("B", "the variable cost (VOM) data agrees with the plant input files",
       "PASS" if not vom_bad else "FAIL", vom_bad[:5] or "modal VOM within 2%, all cases")

# FOM on the mandated tech's new fleet: the smr100 runs build an all-new SMR fleet with one FOM
# value; the large100 runs mix old and new vintages (INFO only).
fom_bad, fom_large = [], {}
for c in CASES:
    itech = META[c]["mandated_tech"]
    sc = load(c, "systemcost_techba")
    fom = sc[(sc["sys_costs"] == "op_fom_costs") & (sc["i"] == itech)].groupby("t")["Value"].sum()
    cap = load(c, "cap")
    capn = cap[cap["i"] == itech].groupby("t")["Value"].sum()
    yrs = [t for t in fom.index if t >= 2038 and capn.get(t, 0) > 0]
    if not yrs:
        continue
    implied = np.mean([fom[t] / capn[t] for t in yrs]) / D2022 / 1000.0
    exp = float(plantchar(META[c]["pc"][itech])["fom"].iloc[-1])
    if META[c]["techscen"] == "smr":
        if abs(implied - exp) / exp > 0.05:
            fom_bad.append((c, round(implied, 1), round(exp, 1)))
    else:
        fom_large[c] = (round(implied, 1), round(exp, 1))
record("B", "the fixed cost (FOM) data agrees with the plant input files (smr100 cases)",
       "PASS" if not fom_bad else "FAIL", fom_bad[:5] or "implied FOM within 5% of plantchar, all smr100 cases")
record("B", "implied large-reactor FOM in the large100 cases (mixed fleet)", "INFO",
       f"(implied, plantchar) $/kW-yr 2022$: {fom_large}")

[PASS] B :: the capital cost data agrees with the plant input files (all 25 cases) -- 2050 (case,tech,year) points; max rel err 4.24e-07 at (smr100_eo_p05, nuclear-smr, 2050)
[PASS] B :: the heat rate data agrees with the plant input files -- both techs, all cases (old vintages can differ a little)


[PASS] B :: the variable cost (VOM) data agrees with the plant input files -- modal VOM within 2%, all cases
[PASS] B :: the fixed cost (FOM) data agrees with the plant input files (smr100 cases) -- implied FOM within 5% of plantchar, all smr100 cases
[INFO] B :: implied large-reactor FOM in the large100 cases (mixed fleet) -- (implied, plantchar) $/kW-yr 2022$: {'large100_eia_p50': (228.1, 171.9), 'large100_aj_p50': (228.7, 178.3), 'large100_iaea_p50': (218.9, 182.0), 'large100_mck_p50': (218.4, 186.9), 'large100_cop28_p50': (194.2, 152.8), 'large100_eo_p50': (178.1, 142.9)}


In [5]:
# ---- B5-B9: financing-multiplier replication (2_financials.gms) -----------------------------
def _resample_schedule(frac, n_years):
    frac = np.asarray(frac, float)
    n0 = len(frac)
    if n_years == n0:
        return frac / frac.sum()
    cdf = np.concatenate([[0.0], np.cumsum(frac)])
    xq = np.linspace(0.0, 1.0, n_years + 1)
    new = np.diff(np.interp(xq, np.linspace(0.0, 1.0, n0 + 1), cdf))
    return new / new.sum()

def ccmult_from_duration(duration_mo, interest_base, canonical_frac):
    n_years = max(1, min(int(round(duration_mo / 12.0)), 10))
    x = _resample_schedule(canonical_frac, n_years)
    return 1.0 + float(np.sum(x * (interest_base ** (np.arange(n_years) + 0.5) - 1.0)))

ccm_check = ccmult_from_duration(72.0, 1.08, np.array([0.1, 0.2, 0.2, 0.2, 0.2, 0.1]))
record("B", "the ccmult check value is correct (QA9 pin)",
       "PASS" if abs(ccm_check - 1.268124) < 1e-5 else "FAIL", f"{ccm_check:.6f}")

fin_cmp, reg_factors = fin_mult_decomposition(CASES, {c: META[c]["fin"] for c in CASES})
fin_cmp.to_csv(S3 / "fin_mult_noITC_comparison.csv", index=False)
record("B", "the finance multiplier has the correct regional structure",
       "PASS" if (fin_cmp.ratio_spread_over_t < 2e-3).all() else "FAIL",
       f"max ratio spread over years {fin_cmp.ratio_spread_over_t.max():.2e} (all 25 cases)")
record("B", "the finance multiplier agrees with our own calculation",
       "PASS" if (fin_cmp.worst_resid_after_regional < 1.5e-3).all() else "FAIL",
       f"worst |residual| {fin_cmp.worst_resid_after_regional.max():.2e} (GAMS rounds to 3 decimals)")
county = pd.read_csv(FIN_DIR / "reg_cap_cost_diff_default.csv")["NUCLEAR"]
lo_b, hi_b = 1 + county.min() - 0.005, 1 + county.max() + 0.005
ok_reg = (((fin_cmp.reg_factor_mean - (1 + county.mean())).abs() < 0.02).all()
          and (fin_cmp.reg_factor_min > lo_b).all() and (fin_cmp.reg_factor_max < hi_b).all())
record("B", "the regional factors are inside the county data range", "PASS" if ok_reg else "FAIL",
       f"implied range [{fin_cmp.reg_factor_min.min():.4f}, {fin_cmp.reg_factor_max.max():.4f}] "
       f"inside county range [{lo_b:.4f}, {hi_b:.4f}]")
xcase = max(float((reg_factors[(c, i)] - reg_factors[(REF, i)]).abs().max())
            for i in NUC for c in CASES)
record("B", "the regional factors are the same in all 25 cases", "PASS" if xcase < 2e-3 else "FAIL",
       f"max cross-case difference {xcase:.2e}")

# ---- B10-B13: no ITC for nuclear --------------------------------------------------------------
worst_wedge, wedge_at = 0.0, None
for c in CASES:
    d = (nuc_slice(c, "cost_cap_fin_mult") - nuc_slice(c, "cost_cap_fin_mult_noITC")).abs()
    if float(d.max()) > worst_wedge:
        worst_wedge, wedge_at = float(d.max()), (c, d.idxmax())
record("B", "the model gives no ITC to nuclear technology (fin_mult == fin_mult_noITC)",
       "PASS" if worst_wedge < 1e-6 else "FAIL",
       f"max |difference| {worst_wedge:.2e}" + (f" at {wedge_at}" if worst_wedge >= 1e-6 else
                                                " in every (tech, region, year) of every case"))
worst_itc, itc_at = 0.0, None
for c in CASES:
    sc = load(c, "systemcost_techba")
    it = sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]
    m = float(it["Value"].abs().max()) if len(it) else 0.0
    if m > worst_itc:
        worst_itc, itc_at = m, c
record("B", "the system cost data shows no ITC payments for nuclear technology",
       "PASS" if worst_itc < 1.0 else "FAIL",
       f"max |nuclear ITC row| ${worst_itc:,.0f}" + (f" in {itc_at}" if worst_itc >= 1.0 else ""))
itc_mag = {c: abs(float(load(c, "tax_expenditure_itc")["Value"].sum())) for c in CASES}
record("B", "the other technologies keep their ITC", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
       f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to "
       f"{max(itc_mag.values())/1e9:.1f} B$ per case (recorded as negative)")
inc_no = pd.read_csv(FIN_DIR / "incentives_obbba_nonuclearitc.csv")
inc_ob = pd.read_csv(FIN_DIR / "incentives_obbba.csv")
nuc_no = inc_no[inc_no["i"].str.upper().str.startswith("NUCLEAR")]
nuc_ob = inc_ob[inc_ob["i"].str.upper().str.startswith("NUCLEAR")]
itc_cols = [c for c in inc_no.columns if c.startswith("itc_")]
ok_in = nuc_no[itc_cols].abs().to_numpy().sum() == 0.0 and nuc_ob["itc_frac"].abs().to_numpy().sum() > 0
record("B", "the incentives input file has zero nuclear ITC (and obbba does not)", "PASS" if ok_in else "FAIL",
       f"{len(nuc_no)} nuclear rows, itc columns all zero; obbba nuclear itc_frac max {nuc_ob['itc_frac'].max():.2f}")

def techs_file(name):
    return [ln.strip() for ln in (NL_DIR / f"nuclear_cap_mandate_techs_{name}.csv").read_text().splitlines()
            if ln.strip() and not ln.startswith("*")]
ok_t = techs_file("smr") == ["nuclear-smr"] and techs_file("large") == ["nuclear"]
record("B", "the mandate technology input files are correct", "PASS" if ok_t else "FAIL",
       f"smr -> {techs_file('smr')}; large -> {techs_file('large')}")

[PASS] B :: the ccmult check value is correct (QA9 pin) -- 1.268124


[PASS] B :: the finance multiplier has the correct regional structure -- max ratio spread over years 7.66e-04 (all 25 cases)
[PASS] B :: the finance multiplier agrees with our own calculation -- worst |residual| 8.35e-04 (GAMS rounds to 3 decimals)
[PASS] B :: the regional factors are inside the county data range -- implied range [0.9528, 1.2539] inside county range [0.9456, 1.2717]
[PASS] B :: the regional factors are the same in all 25 cases -- max cross-case difference 5.11e-04


[PASS] B :: the model gives no ITC to nuclear technology (fin_mult == fin_mult_noITC) -- max |difference| 0.00e+00 in every (tech, region, year) of every case
[PASS] B :: the system cost data shows no ITC payments for nuclear technology -- max |nuclear ITC row| $0
[PASS] B :: the other technologies keep their ITC -- total ITC tax expenditure magnitude 123.5 to 124.8 B$ per case (recorded as negative)


[PASS] B :: the incentives input file has zero nuclear ITC (and obbba does not) -- 8 nuclear rows, itc columns all zero; obbba nuclear itc_frac max 0.30
[PASS] B :: the mandate technology input files are correct -- smr -> ['nuclear-smr']; large -> ['nuclear']


### Phase C: mandate mechanics and the shadow-price table

`duals_by_year.csv` has one row per run and model year: `mandate_MW` (the target of the run's
trajectory file), `cap_MW` (national capacity of the mandated tech, rounded to 0.1 MW), `slack_MW`,
`dual_2004_MWyr` (the shadow price, 2004 $/MW-yr), `dual_raw` (objective units) and `dual_2024_MWyr`.
C1 to C3 test the floor, binding and slack conditions on the unrounded capacity; C4 audits the
conversion `raw = cost_scale x pvf_onm x converted`; C5 to C7 test that each run builds only its own
technology and no SMR before 2031. `overbuild_by_case.csv` counts the binding and slack years.

In [6]:
TRAJ = {c: trajectory(META[c]["scen"]) for c in CASES}
DUAL, RAW = {}, {}
for c in CASES:
    DUAL[c], RAW[c] = dual_series(c)
MAND_CAP = {c: (lambda cap: cap[cap["i"] == META[c]["mandated_tech"]].groupby("t")["Value"].sum())(load(c, "cap"))
            for c in CASES}

duals_tbl, viol, bind_bad, slack_bad = duals_table(CASES, YEARS_RUN, TRAJ, MAND_CAP, DUAL, RAW)
duals_tbl.to_csv(S3 / "duals_by_year.csv", index=False)
assert len(duals_tbl) == 25 * len(YEARS_RUN) == 400, "25 runs x 16 model years"
record("C", "the capacity satisfies the mandate floor in every case", "PASS" if not viol else "FAIL",
       viol[:5] or "no violation larger than 5 MW in any mandated year")
record("C", "the capacity equals the floor in each binding year (positive dual)",
       "PASS" if not bind_bad else "FAIL", bind_bad[:5] or "every year with dual > $1/MW-yr has |slack| <= 5 MW")
record("C", "a positive dual occurs only in the binding years", "PASS" if not slack_bad else "FAIL",
       slack_bad[:5] or "every year with slack > 100 MW has a zero dual")
gate_dual_conversion(record, "C", CASES, DUAL, RAW)

bad_large = [(c, float(v)) for c in SMR_CASES
             if (v := load(c, "cap_new_ann").query("i == 'nuclear' and t > 2030")["Value"].sum()) > 0]
bad_smr = [(c, float(v)) for c in LARGE_CASES
           if (v := load(c, "cap_new_ann").query("i == 'nuclear-smr'")["Value"].sum()) > 0]
record("C", "no large-reactor builds occur after 2030 in the smr100 cases", "PASS" if not bad_large else "FAIL",
       bad_large or "all 19 smr100 cases clean")
record("C", "no SMR builds occur in the large100 cases", "PASS" if not bad_smr else "FAIL",
       bad_smr or "all 6 large100 cases clean")
pre = {c: float(load(c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in SMR_CASES}
record("C", "no SMR capacity exists before 2031", "PASS" if max(pre.values()) == 0 else "FAIL",
       f"max pre-2031 SMR capacity {max(pre.values()):.1f} MW")
smr_trap = [t for t in set(load(REF, "cap")["i"].unique()) if "smr" in t.lower() and t != "nuclear-smr"]
record("C", "the technology names are correct (no stray SMR names)", "PASS" if not smr_trap else "INFO",
       smr_trap or "tech set clean; nuclear filters use the exact name nuclear-smr (SMR elsewhere = steam methane reforming)")

ob = overbuild_table(duals_tbl, CASES, YEARS_RUN, TRAJ)
ob.to_csv(S3 / "overbuild_by_case.csv", index=False)
record("C", "overbuild and binding years by case", "INFO",
       f"binding-year counts {dict(zip(ob.case, ob.n_binding_years))}")
display(ob)

[PASS] C :: the capacity satisfies the mandate floor in every case -- no violation larger than 5 MW in any mandated year
[PASS] C :: the capacity equals the floor in each binding year (positive dual) -- every year with dual > $1/MW-yr has |slack| <= 5 MW
[PASS] C :: a positive dual occurs only in the binding years -- every year with slack > 100 MW has a zero dual
[PASS] C :: the dual conversion is correct (raw = cost_scale x pvf_onm x converted) -- max rel err 1.29e-07 across all cases with duals
[PASS] C :: no large-reactor builds occur after 2030 in the smr100 cases -- all 19 smr100 cases clean
[PASS] C :: no SMR builds occur in the large100 cases -- all 6 large100 cases clean


[PASS] C :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW
[PASS] C :: the technology names are correct (no stray SMR names) -- tech set clean; nuclear filters use the exact name nuclear-smr (SMR elsewhere = steam methane reforming)
[INFO] C :: overbuild and binding years by case -- binding-year counts {'smr100_eia_p05': 4, 'smr100_eia_p50': 5, 'smr100_eia_p95': 5, 'smr100_aj_p05': 10, 'smr100_aj_p50': 10, 'smr100_aj_p95': 10, 'smr100_iaea_p05': 10, 'smr100_iaea_p50': 10, 'smr100_iaea_p95': 10, 'smr100_mck_p05': 10, 'smr100_mck_p50': 10, 'smr100_mck_p95': 10, 'smr100_cop28_p05': 10, 'smr100_cop28_p50': 10, 'smr100_cop28_p95': 10, 'smr100_eo_p05': 10, 'smr100_eo_p50': 10, 'smr100_eo_p95': 10, 'large100_eia_p50': 6, 'large100_aj_p50': 10, 'large100_iaea_p50': 10, 'large100_mck_p50': 10, 'large100_cop28_p50': 10, 'large100_eo_p50': 10, 'smr100_eia_p50_eq': 5}


,case,n_mandated_years,n_binding_years,n_slack_years,first_binding_year,max_overbuild_ratio
0,smr100_eia_p05,5,4,1,2038,1.03
1,smr100_eia_p50,5,5,0,2038,1.00
2,smr100_eia_p95,5,5,0,2038,1.00
3,smr100_aj_p05,10,10,0,2031,1.00
4,smr100_aj_p50,10,10,0,2031,1.00
5,smr100_aj_p95,10,10,0,2031,1.00
6,smr100_iaea_p05,10,10,0,2031,1.00
7,smr100_iaea_p50,10,10,0,2031,1.00
8,smr100_iaea_p95,10,10,0,2031,1.00
9,smr100_mck_p05,10,10,0,2031,1.00


### Phase D: the equality twin

`smr100_eia_p50_eq` imposes the target as an equality (floor and ceiling) with the inputs of
`smr100_eia_p50`. The same objective, the same floor shadow price and a zero ceiling shadow price
show that the constraint form changes nothing. Regional siting can differ between equally optimal
solutions, so the comparison uses national totals. `eq_flip_comparison.csv` gives the largest
difference of national capacity and generation by (tech, year).

In [7]:
EQ, P50 = "smr100_eia_p50_eq", "smr100_eia_p50"
d_obj = abs(obj[EQ] - obj[P50]) / obj[P50]
record("D", "the equality case gives the same objective value", "PASS" if d_obj < 1e-6 else "FAIL",
       f"rel difference {d_obj:.2e} ({obj[EQ]:.6e} vs {obj[P50]:.6e})")
eq_rows, worst_nat = [], {}
for key in ["cap", "gen_ann"]:
    a = load(EQ, key).groupby(["i", "t"])["Value"].sum()
    b = load(P50, key).groupby(["i", "t"])["Value"].sum()
    j = pd.concat([a, b], axis=1, keys=["eq", "p50"]).fillna(0.0)
    d = (j["eq"] - j["p50"]).abs()
    rel = d / j["p50"].abs().clip(lower=1.0)
    worst_nat[key] = float(rel.max())
    eq_rows.append(dict(dataset=key, max_abs_diff=float(d.max()), max_rel_diff=float(rel.max()),
                        worst_at=str(d.idxmax())))
eqf = pd.DataFrame(eq_rows)
eqf.to_csv(S3 / "eq_flip_comparison.csv", index=False)
record("D", "the equality case gives the same national capacity and generation",
       "PASS" if max(worst_nat.values()) < 1e-3 else "INFO", {k: f"{v:.2e}" for k, v in worst_nat.items()})
idx = sorted(set(DUAL[EQ].index) | set(DUAL[P50].index))
de, dp = DUAL[EQ].reindex(idx).fillna(0.0), DUAL[P50].reindex(idx).fillna(0.0)
rel_d = float((de - dp).abs().max() / dp.max()) if dp.max() > 0 else 0.0
record("D", "the equality case gives the same floor dual values", "PASS" if rel_d < 1e-3 else "FAIL",
       f"max rel difference {rel_d:.2e} over years {idx}")
ub_present = [k for k in ["nuclear_cap_price_ub", "nuclear_cap_price_ub_raw"] if k in KEYS[EQ]]
record("D", "the ceiling dual is zero in the equality case", "PASS" if not ub_present else "INFO",
       "ceiling dual keys absent = all zero (zero-suppression rule)" if not ub_present
       else f"ceiling dual present: {ub_present}")
verdict = d_obj < 1e-6 and rel_d < 1e-3 and not ub_present
record("D", "verdict: the constraint form causes no change in behavior", "PASS" if verdict else "FAIL",
       "floor results stand" if verdict else "differences found; see D1-D4")
display(eqf)

[PASS] D :: the equality case gives the same objective value -- rel difference 0.00e+00 (1.631947e+12 vs 1.631947e+12)


[INFO] D :: the equality case gives the same national capacity and generation -- {'cap': '4.38e-04', 'gen_ann': '2.78e-02'}
[PASS] D :: the equality case gives the same floor dual values -- max rel difference 1.67e-05 over years [2038, 2041, 2044, 2047, 2050]
[PASS] D :: the ceiling dual is zero in the equality case -- ceiling dual keys absent = all zero (zero-suppression rule)
[PASS] D :: verdict: the constraint form causes no change in behavior -- floor results stand


,dataset,max_abs_diff,max_rel_diff,worst_at
0,cap,3.632812,0.000438,"('wind-ons_9', 2050)"
1,gen_ann,600696.000000,0.027753,"('wind-ofs_10', 2041)"


### Phase E: load integrity

All 25 runs use one demand scenario, so the exogenous load (busbar load `load_rt`, timeslice hours,
the `end_use` and `dist_loss` rows of `load_cat`) must be identical across runs
(`load_invariance.csv`). Storage charging, hydrogen production, losses and the stress-period load
move with each solution and are context only. E3 tests dropped load after 2025, E4 the demand input
table, E6 the load against the old pilot run (the load inputs did not change between the two
vintages).

In [8]:
def frame(case, key, cat=None):
    df = load(case, key)
    if cat is not None:
        df = df[df["loadtype"] == cat]
    return df.set_index([c for c in df.columns if c != "Value"])["Value"]

li_rows = []
for key, cat in [("load_rt", None), ("hours", None), ("load_cat", "end_use"), ("load_cat", "dist_loss")]:
    ref_s = frame(REF, key, cat)
    for c in CASES:
        if c == REF:
            continue
        j = pd.concat([ref_s, frame(c, key, cat)], axis=1, keys=["ref", "c"]).fillna(0.0)
        d = (j["ref"] - j["c"]).abs()
        li_rows.append(dict(dataset=key + (f"[{cat}]" if cat else ""), case=c, max_abs_diff=float(d.max()),
                            max_rel_diff=float((d / j["ref"].abs().clip(lower=1.0)).max())))
li = pd.DataFrame(li_rows)
li.to_csv(S3 / "load_invariance.csv", index=False)
w = li.loc[li.max_rel_diff.idxmax()]
record("E", "the exogenous load data is the same in all 25 cases",
       "PASS" if li.max_rel_diff.max() < 1e-6 else "FAIL",
       f"load_rt, hours, load_cat[end_use], load_cat[dist_loss] vs {REF}; max rel difference "
       f"{li.max_rel_diff.max():.2e} ({w.dataset}, {w.case})")

endo = {}
for key, cat in [("load_cat", "stor_charge"), ("load_cat", "h2_prod"), ("load_cat", "trans_loss"),
                 ("load_stress", None)]:
    ref_s = frame(REF, key, cat)
    worst = 0.0
    for c in CASES:
        if c == REF:
            continue
        j = pd.concat([ref_s, frame(c, key, cat)], axis=1, keys=["ref", "c"]).fillna(0.0)
        worst = max(worst, abs(float(j["c"].sum()) - float(j["ref"].sum())) / max(float(j["ref"].sum()), 1.0))
    endo[key + (f"[{cat}]" if cat else "")] = round(worst, 4)
record("E", "the endogenous load parts differ only a little between cases", "INFO",
       f"max rel difference of national totals vs {REF}: {endo} (these parts move with each solution)")
gate_dropped_load(record, "E", CASES)

dem = pd.read_csv(INPUTS / "load" / "demand_AEO_2026_baseline.csv")
ok_dem = (not dem.isna().any().any() and (dem["multiplier"] > 0).all()
          and dem["multiplier"].between(0.3, 5.0).all() and dem["year"].min() <= 2010
          and dem["year"].max() >= 2050 and dem.groupby("year")["r"].count().nunique() == 1)
record("E", "the demand input file is complete and correct", "PASS" if ok_dem else "FAIL",
       f"{dem['r'].nunique()} states x years {dem['year'].min()}-{dem['year'].max()}; multiplier range "
       f"[{dem['multiplier'].min():.3f}, {dem['multiplier'].max():.3f}]")
record("E", "the hourly load profiles are complete and correct", "INFO",
       "dropped in the port: an integrity scan of the HDF5 input profiles (inputs/profiles_demand/*.h5), "
       "not of a run output; E1 and E6 test the load the runs actually used", port="moved")

pl = load("smr100_eia_mid", "load_rt").set_index(["r", "t"])["Value"]
pr = load(REF, "load_rt").set_index(["r", "t"])["Value"]
shared = pl.index.intersection(pr.index)
rel = float(((pl.loc[shared] - pr.loc[shared]).abs() / pl.loc[shared].abs().clip(lower=1.0)).max())
record("E", "the output load agrees with the old pilot runs on the shared years", "PASS" if rel < 1e-6 else "FAIL",
       f"{len(shared)} shared (region, year) points; max rel difference {rel:.2e}")

nat = load(REF, "load_rt").groupby("t")["Value"].sum()
gen = load(REF, "gen_ann").groupby("t")["Value"].sum()
mult = dem[dem["year"].isin([2026, 2050])].groupby("year")["multiplier"].mean()
record("E", "the output load grows like the demand scenario", "INFO",
       f"national load 2026->2050 x{nat.loc[2050] / nat.loc[2026]:.2f}; demand multiplier mean "
       f"x{mult.loc[2050] / mult.loc[2026]:.2f}")
ratio = (gen / nat).loc[[t for t in YEARS_RUN if t >= 2026]]
record("E", "generation covers the load in every model year", "INFO",
       f"generation / busbar load range [{ratio.min():.3f}, {ratio.max():.3f}]")

[PASS] E :: the exogenous load data is the same in all 25 cases -- load_rt, hours, load_cat[end_use], load_cat[dist_loss] vs smr100_eia_p50; max rel difference 0.00e+00 (load_rt, smr100_eia_p05)


[INFO] E :: the endogenous load parts differ only a little between cases -- max rel difference of national totals vs smr100_eia_p50: {'load_cat[stor_charge]': 0.0423, 'load_cat[h2_prod]': 0.7131, 'load_cat[trans_loss]': 0.0488, 'load_stress': 0.0107} (these parts move with each solution)
[PASS] E :: no load is dropped after 2025 -- all 25 cases clean; historical 2010-2023 artifact 2.96 TWh, case spread 0.0 MWh
[PASS] E :: the demand input file is complete and correct -- 48 states x years 2010-2050; multiplier range [0.763, 2.785]
[INFO] E :: the hourly load profiles are complete and correct -- dropped in the port: an integrity scan of the HDF5 input profiles (inputs/profiles_demand/*.h5), not of a run output; E1 and E6 test the load the runs actually used
[PASS] E :: the output load agrees with the old pilot runs on the shared years -- 1170 shared (region, year) points; max rel difference 0.00e+00
[INFO] E :: the output load grows like the demand scenario -- national load 2026->2050 x1

### Phase F: cross-case gates

F1: within each schedule the shadow price must not cross between the p05, p50 and p95 cost draws (a
more expensive world needs a larger subsidy; tolerance $50/MW-yr + 0.1 %). F2: the objective values
keep the same order. F3/F4 scan every table for NaN and wrong negatives (key inventory), F5 tests
national totals, F6 to F8 record context.

In [9]:
mono_bad = []
for tok in SCHEDULES:
    d05, d50, d95 = (DUAL[f"smr100_{tok}_{p}"] for p in ["p05", "p50", "p95"])
    idx = sorted(set(d05.index) | set(d50.index) | set(d95.index))
    d05, d50, d95 = (s.reindex(idx).fillna(0.0) for s in (d05, d50, d95))
    tol = 50.0 + 0.001 * d95.abs()
    bad_years = [int(t) for t in idx if d95[t] < d50[t] - tol[t] or d50[t] < d05[t] - tol[t]]
    if bad_years:
        mono_bad.append((tok, bad_years))
record("F", "the dual values do not cross between p05, p50, and p95 (all 6 schedules)",
       "PASS" if not mono_bad else "FAIL", mono_bad or "p95 >= p50 >= p05 in every year of every schedule")
ord_bad = [(tok, o) for tok in SCHEDULES
           if not ((o := [obj[f"smr100_{tok}_{p}"] for p in ["p05", "p50", "p95"]])[0] <= o[1] <= o[2])]
record("F", "the objective values are in the correct order (p05 <= p50 <= p95)",
       "PASS" if not ord_bad else "FAIL", ord_bad or "a more expensive world always costs more, all 6 schedules")
gate_data_scan(record, "F", CASES)
gate_national_totals(record, "F", CASES)

decay = {}
for c in CASES:
    dd = DUAL[c][DUAL[c] > 1.0].sort_index()
    if len(dd) >= 2:
        decay[c] = dict(peak_year=int(dd.idxmax()), end_over_peak=round(float(dd.iloc[-1] / dd.max()), 2))
record("F", "dual decay shape by case (bridge hypothesis, first look)", "INFO", f"peak year + end/peak ratio: {decay}")
ladder = {}
for tok in SCHEDULES:
    d = DUAL[f"smr100_{tok}_p50"]
    ladder[tok] = dict(n_binding=int((d > 1.0).sum()),
                       mean_dual_2024=round(float(d[d > 1.0].mean() * TO2024)) if (d > 1.0).any() else 0)
record("F", "dual level by schedule ambition (p50 cases)", "INFO", ladder)
l_vs_s = {}
for tok in SCHEDULES:
    dl_, ds_ = DUAL[f"large100_{tok}_p50"], DUAL[f"smr100_{tok}_p50"]
    shared = sorted(set(dl_[dl_ > 1.0].index) & set(ds_[ds_ > 1.0].index))
    if shared:
        l_vs_s[tok] = round(float(dl_.reindex(shared).mean() / ds_.reindex(shared).mean()), 2)
record("F", "large100 dual level over smr100 dual level (shared binding years)", "INFO",
       f"mean ratio by schedule: {l_vs_s}")

summary3 = write_registry("step3_checks")

[PASS] F :: the dual values do not cross between p05, p50, and p95 (all 6 schedules) -- p95 >= p50 >= p05 in every year of every schedule
[PASS] F :: the objective values are in the correct order (p05 <= p50 <= p95) -- a more expensive world always costs more, all 6 schedules


[PASS] F :: no data value is NaN or infinite in any file -- all 209 keys x 25 files scanned (key_inventory.n_nonfinite)
[PASS] F :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (key_inventory.min_*)
[PASS] F :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 25 cases (storage charging and upgrades make some non-nuclear rows negative; a normal convention)


[PASS] F :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 25 cases within 1e-3
[INFO] F :: dual decay shape by case (bridge hypothesis, first look) -- peak year + end/peak ratio: {'smr100_eia_p05': {'peak_year': 2038, 'end_over_peak': 0.03}, 'smr100_eia_p50': {'peak_year': 2038, 'end_over_peak': 0.76}, 'smr100_eia_p95': {'peak_year': 2038, 'end_over_peak': 0.84}, 'smr100_aj_p05': {'peak_year': 2031, 'end_over_peak': 0.19}, 'smr100_aj_p50': {'peak_year': 2033, 'end_over_peak': 0.83}, 'smr100_aj_p95': {'peak_year': 2033, 'end_over_peak': 0.83}, 'smr100_iaea_p05': {'peak_year': 2031, 'end_over_peak': 0.19}, 'smr100_iaea_p50': {'peak_year': 2031, 'end_over_peak': 0.31}, 'smr100_iaea_p95': {'peak_year': 2031, 'end_over_peak': 0.82}, 'smr100_mck_p05': {'peak_year': 2031, 'end_over_peak': 0.13}, 'smr100_mck_p50': {'peak_year': 2031, 'end_over_peak': 0.46}, 'smr100_mck_p95': {'peak_year': 2031, 'end_over_peak': 0.77}, 'smr100_cop28_p05': {'pe

## 3. The step4 batch (`step4_checks`): 108 market-sensitivity runs and 12 large-reactor percentile runs

The 108 sensitivity runs are the 18 `smr100` base runs, each under one of six market switches:
`gaslo` (high oil and gas supply, so a **low** gas price), `gashi`, `demhi` (high electrification
demand), `relo` / `rehi` (low / high renewable capital cost), `translim` (limited new
transmission). Their nuclear inputs are the base run's files. The 12 large-reactor runs add the p05
and p95 cost draws to the `smr100` batch's `large100_*_p50` runs.

Derived tables and readers: `duals_by_year.csv` (the six market worlds' shadow prices; read by 07, 08,
11 and 13a/13b) and `overbuild_by_case.csv` (read by 07 and 08). The step-3 base-run shadow prices
stay in `step3_checks/duals_by_year.csv`. The evidence tables of the B gates are
`pointer_copy_comparison.csv` (nuclear inputs identical to the base run), `switch_echo.csv` (each
market switch moved its own input), `fingerprint_errors.csv` and `fin_mult_noITC_comparison.csv`
(the 12 large-reactor runs) and `load_invariance.csv`.

In [10]:
record = registry("step4_checks")
S4 = out_dir("step4_checks")

cases4 = pd.read_csv(paths.casefile("step4"), index_col=0)
CASES = [c for c in cases4.columns if c != "Default Value"]
assert len(CASES) == 120, f"the step4 casefile has 120 runs, got {len(CASES)}"
CASES3 = [c for c in cases_csv.columns if c != "Default Value"]
sw = lambda c, n: switch(cases_csv if c in cases_csv.columns else cases4, c, n)
SENS = ["gaslo", "gashi", "demhi", "relo", "rehi", "translim"]

META = {}
for c in CASES:
    parts = c.split("_")
    fam, sens = ("large" if c.startswith("large100") else "smr"), (parts[3] if len(parts) > 3 else "")
    ts = sw(c, "GSw_NuclearCapMandateTechScen")
    META[c] = dict(fam=fam, sched=parts[1], pct=parts[2], sens=sens,
                   scen=sw(c, "GSw_NuclearCapMandateScen"), techscen=ts,
                   mandated_tech={"smr": "nuclear-smr", "large": "nuclear"}[ts],
                   pc={"nuclear": sw(c, "plantchar_nuclear"), "nuclear-smr": sw(c, "plantchar_nuclear_smr")},
                   fin=sw(c, "financials_tech_suffix"),
                   base="_".join(parts[:3]) if sens else None)
SENS_CASES = [c for c in CASES if META[c]["sens"]]
LARGE_CASES = [c for c in CASES if META[c]["fam"] == "large"]
assert len(SENS_CASES) == 108 and len(LARGE_CASES) == 12
BASE_CASES = sorted({META[c]["base"] for c in SENS_CASES})
assert len(BASE_CASES) == 18 and all(b in CASES3 for b in BASE_CASES)
LARGE_P50 = [f"large100_{tok}_p50" for tok in SCHEDULES]
REF = "smr100_eia_p50"

# ---- phase A ----------------------------------------------------------------------------------
gate_census(record, "A", "step4", CASES, "one output file exists for each of the 120 cases")
gate_census(record, "A", "smr100", sorted(set(BASE_CASES) | set(LARGE_P50) | {REF}),
            "the Step 3 base-case files are available for comparison")
extra = sorted(set(INV.loc[INV.batch == "step4", "case"]) - set(CASES))
record("A", "no unexpected files are in the folder", "PASS" if not extra else "INFO",
       extra or "every step4 run in the archive is a casefile column (key_inventory)", port="adapted")
record("A", "the delivery archive was extracted and verified", "INFO",
       "dropped in the port: zip extraction and SHA verification; data/reeds_outputs_manifest.csv holds "
       "every file's SHA-256", port="moved")
keysets, union = gate_keys(record, "A", CASES,
                           "all 120 files have the same data keys (dual keys can be absent when zero)",
                           ref=REF, ref_check="the key set agrees with the Step 3 vintage")
no_dual = sorted(c for c in CASES if "nuclear_cap_price" not in KEYS[c])
record("A", "cases with no dual key (all-zero dual, possibly unbound)", "INFO",
       no_dual or "every case has a dual key")
EXPECT_YEARS = [int(y) for y in sw(REF, "yearset").split("_")]
YEARS_RUN = EXPECT_YEARS
obj = gate_solve_health(record, "A", CASES, EXPECT_YEARS)
for b in set(BASE_CASES) | set(LARGE_P50):
    obj[b] = float(load(b, "objfn_raw")["Value"].iloc[0])

[PASS] A :: one output file exists for each of the 120 cases -- 120 runs returned (key_inventory: batch step4)
[PASS] A :: the Step 3 base-case files are available for comparison -- 24 runs returned (key_inventory: batch smr100)
[PASS] A :: no unexpected files are in the folder -- every step4 run in the archive is a casefile column (key_inventory)
[INFO] A :: the delivery archive was extracted and verified -- dropped in the port: zip extraction and SHA verification; data/reeds_outputs_manifest.csv holds every file's SHA-256
[PASS] A :: all 120 files have the same data keys (dual keys can be absent when zero) -- key counts [209]; runs without a shadow-price table: 0
[PASS] A :: the key set agrees with the Step 3 vintage -- 209 keys, no drift
[INFO] A :: cases with no dual key (all-zero dual, possibly unbound) -- every case has a dual key


[PASS] A :: the solver residuals are small in every file -- worst |residual| 1.04e-03 at smr100_iaea_p05_gashi


[PASS] A :: the objective value is a normal positive number in every file -- range 1.2971e+12 to 2.2344e+12


[PASS] A :: the model years are correct in every file (annual 2031-2035 block present) -- expected [2010, 2015, 2020, 2023, 2026, 2029, 2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]


[PASS] A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 at 14.7645 (max dev 9.5e-07); cost_scale==1; z_rep has 16 years in every file


### Phase B: pointer-copy identity, no-ITC baseline, market-switch echoes, large-reactor fingerprints

B1: the nuclear capital cost and financing multiplier of each sensitivity run equal its base run's
(a stronger test than a fingerprint, and a proof of no input drift between the two NREL deliveries);
no run gives an ITC to nuclear. B2: each market switch moved its own input in the intended direction
and did not leak into the other arms (only the exogenous load and the renewable capital costs can
carry a leak test; the gas price is an equilibrium output that every arm moves). B3: the Step 3
fingerprint gates on the 12 new large-reactor runs.

In [11]:
pc = input_identity(SENS_CASES, {c: META[c]["base"] for c in SENS_CASES}).rename(columns={})
pc.to_csv(S4 / "pointer_copy_comparison.csv", index=False)
worst_cc = pc[pc.key == "cost_cap"].max_rel_diff.max()
worst_fm = pc[pc.key == "cost_cap_fin_mult_noITC"].max_rel_diff.max()
record("B1", "the nuclear capital cost is identical to the base-case output (108 cases)",
       "PASS" if worst_cc < 1e-6 else "FAIL", f"max rel difference {worst_cc:.2e}")
record("B1", "the nuclear finance multiplier is identical to the base-case output (108 cases)",
       "PASS" if worst_fm < 1e-6 else "FAIL", f"max rel difference {worst_fm:.2e}")

worst_wedge, wedge_at = 0.0, None
for c in CASES:
    m = float((nuc_slice(c, "cost_cap_fin_mult") - nuc_slice(c, "cost_cap_fin_mult_noITC")).abs().max())
    if m > worst_wedge:
        worst_wedge, wedge_at = m, c
record("B1", "the model gives no ITC to nuclear technology (fin_mult == fin_mult_noITC, all 120)",
       "PASS" if worst_wedge < 1e-6 else "FAIL",
       f"max |difference| {worst_wedge:.2e}" + (f" at {wedge_at}" if worst_wedge >= 1e-6 else ""))
worst_itc, itc_at = 0.0, None
for c in CASES:
    sc = load(c, "systemcost_techba")
    it = sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]
    m = float(it["Value"].abs().max()) if len(it) else 0.0
    if m > worst_itc:
        worst_itc, itc_at = m, c
record("B1", "the system cost data shows no ITC payments for nuclear technology (all 120)",
       "PASS" if worst_itc < 1.0 else "FAIL",
       f"max |nuclear ITC row| ${worst_itc:,.0f}" + (f" in {itc_at}" if worst_itc >= 1.0 else ""))
itc_mag = {c: abs(float(load(c, "tax_expenditure_itc")["Value"].sum())) for c in CASES}
record("B1", "the other technologies keep their ITC (all 120)", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
       f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

[PASS] B1 :: the nuclear capital cost is identical to the base-case output (108 cases) -- max rel difference 0.00e+00
[PASS] B1 :: the nuclear finance multiplier is identical to the base-case output (108 cases) -- max rel difference 0.00e+00


[PASS] B1 :: the model gives no ITC to nuclear technology (fin_mult == fin_mult_noITC, all 120) -- max |difference| 0.00e+00


[PASS] B1 :: the system cost data shows no ITC payments for nuclear technology (all 120) -- max |nuclear ITC row| $0
[PASS] B1 :: the other technologies keep their ITC (all 120) -- total ITC tax expenditure magnitude 109.8 to 146.4 B$ per case


In [12]:
echo_rows = []

def nat_series(case, key):
    return load(case, key).groupby("t")["Value"].sum()

gas_bad = []
for c in SENS_CASES:
    s = META[c]["sens"]
    if s not in ("gaslo", "gashi"):
        continue
    g = load(c, "repgasprice_nat").set_index("t")["Value"]
    gb = load(META[c]["base"], "repgasprice_nat").set_index("t")["Value"]
    yrs = [t for t in g.index if t >= 2026 and t in gb.index]
    d = g.reindex(yrs) - gb.reindex(yrs)
    frac_dir = float((d < 0).mean()) if s == "gaslo" else float((d > 0).mean())
    echo_rows.append(dict(case=c, sens=s, metric="gas_price_mean_rel",
                          value=float((g.reindex(yrs) / gb.reindex(yrs)).mean())))
    if frac_dir < 0.9:
        gas_bad.append((c, round(frac_dir, 2)))
record("B2", "the gas price sits below the base in gaslo and above it in gashi", "PASS" if not gas_bad else "FAIL",
       gas_bad[:5] or "correct direction in >=90% of model years, all 36 gas cases")

dem_bad = []
DEMHI = [c for c in SENS_CASES if META[c]["sens"] == "demhi"]
for c in DEMHI:
    lr, lb = nat_series(c, "load_rt"), nat_series(META[c]["base"], "load_rt")
    yrs = [t for t in lr.index if t >= 2031]
    echo_rows.append(dict(case=c, sens="demhi", metric="load_mean_rel",
                          value=float((lr.reindex(yrs) / lb.reindex(yrs)).mean())))
    if not (lr.reindex(yrs) >= lb.reindex(yrs) - 1.0).all():
        dem_bad.append(c)
record("B2", "the exogenous load sits above the base case in every demhi case", "PASS" if not dem_bad else "FAIL",
       dem_bad[:5] or f"18 demhi cases; mean load ratio "
       f"{np.mean([r['value'] for r in echo_rows if r['metric'] == 'load_mean_rel']):.3f}")
ref_l = load(DEMHI[0], "load_rt").set_index(["r", "t"])["Value"]
dem_same = 0.0
for c in DEMHI[1:]:
    j = pd.concat([ref_l, load(c, "load_rt").set_index(["r", "t"])["Value"]], axis=1, keys=["a", "b"]).fillna(0.0)
    dem_same = max(dem_same, float(((j["a"] - j["b"]).abs() / j["a"].abs().clip(lower=1.0)).max()))
record("B2", "the exogenous load is identical inside the demhi block", "PASS" if dem_same < 1e-6 else "FAIL",
       f"max rel difference {dem_same:.2e}")

RE_FAMS = ["upv", "wind-ons", "wind-ofs", "battery", "csp", "geo"]

def re_cost(case):
    cc = load(case, "cost_cap")
    cc = cc.assign(fam=cc["i"].str.extract(r"^([a-z\-]+)", expand=False))
    cc = cc[cc["fam"].isin(RE_FAMS) & (cc["t"] >= 2026)]
    return cc.groupby(["fam", "t"])["Value"].mean()

re_bad = []
for c in SENS_CASES:
    s = META[c]["sens"]
    if s not in ("relo", "rehi"):
        continue
    j = pd.concat([re_cost(c), re_cost(META[c]["base"])], axis=1, keys=["sens", "base"]).dropna()
    rel = j["sens"] / j["base"]
    frac_dir = float((rel <= 1.0 + 1e-9).mean()) if s == "relo" else float((rel >= 1.0 - 1e-9).mean())
    echo_rows.append(dict(case=c, sens=s, metric="re_cost_mean_rel", value=float(rel.mean())))
    if frac_dir < 0.9:
        re_bad.append((c, round(frac_dir, 2), round(float(rel.mean()), 3)))
record("B2", "the renewable capital costs sit below the base in relo and above it in rehi",
       "PASS" if not re_bad else "FAIL", re_bad[:5] or "correct direction in >=90% of (family, year) points, all 36 RE cases")

tr_bad = []
for c in SENS_CASES:
    if META[c]["sens"] != "translim":
        continue
    a = float(load(c, "invtran_out")["Value"].sum())
    b = float(load(META[c]["base"], "invtran_out")["Value"].sum())
    echo_rows.append(dict(case=c, sens="translim", metric="new_transmission_rel", value=a / b if b else np.nan))
    if a > b * 1.01:
        tr_bad.append((c, round(a / b, 3)))
record("B2", "the new transmission build does not go above the base case in translim",
       "PASS" if not tr_bad else "FAIL",
       tr_bad[:5] or "national new-transmission totals <= base, all 18 translim cases (national totals only)")

leak, gas_move = [], {}
for c in SENS_CASES:
    s, b = META[c]["sens"], META[c]["base"]
    if s not in ("gaslo", "gashi"):
        g = load(c, "repgasprice_nat").set_index("t")["Value"]
        gb = load(b, "repgasprice_nat").set_index("t")["Value"]
        gas_move.setdefault(s, []).append(float(((g - gb).abs() / gb.abs().clip(lower=1e-9)).max()))
    if s != "demhi":
        lr, lb = nat_series(c, "load_rt"), nat_series(b, "load_rt")
        rel = float(((lr - lb).abs() / lb.abs().clip(lower=1.0)).max())
        if rel > 1e-6:
            leak.append((c, "load_rt", round(rel, 6)))
    if s not in ("relo", "rehi"):
        j = pd.concat([re_cost(c), re_cost(b)], axis=1, keys=["a", "b"]).dropna()
        rel = float(((j["a"] - j["b"]).abs() / j["b"].abs().clip(lower=1e-9)).max())
        if rel > 1e-6:
            leak.append((c, "re_cost", round(rel, 6)))
record("B2", "no market switch leaks into the other arms (exogenous inputs only)", "PASS" if not leak else "FAIL",
       leak[:8] or "exogenous load and RE capital costs identical to base outside their arms")
record("B2", "endogenous gas-price response outside the gas arms", "INFO",
       {s: f"max {max(v):.3f}" for s, v in sorted(gas_move.items())})
pd.DataFrame(echo_rows).to_csv(S4 / "switch_echo.csv", index=False)

[PASS] B2 :: the gas price sits below the base in gaslo and above it in gashi -- correct direction in >=90% of model years, all 36 gas cases


[PASS] B2 :: the exogenous load sits above the base case in every demhi case -- 18 demhi cases; mean load ratio 1.156
[PASS] B2 :: the exogenous load is identical inside the demhi block -- max rel difference 0.00e+00


[PASS] B2 :: the renewable capital costs sit below the base in relo and above it in rehi -- correct direction in >=90% of (family, year) points, all 36 RE cases
[PASS] B2 :: the new transmission build does not go above the base case in translim -- national new-transmission totals <= base, all 18 translim cases (national totals only)


[PASS] B2 :: no market switch leaks into the other arms (exogenous inputs only) -- exogenous load and RE capital costs identical to base outside their arms
[INFO] B2 :: endogenous gas-price response outside the gas arms -- {'demhi': 'max 0.187', 'rehi': 'max 0.147', 'relo': 'max 0.130', 'translim': 'max 0.050'}


In [13]:
fp4 = cost_fingerprint(LARGE_CASES, {c: META[c]["pc"] for c in LARGE_CASES})
fp4.to_csv(S4 / "fingerprint_errors.csv", index=False)
w = fp4.loc[fp4.rel_err.idxmax()]
record("B3", "the capital cost data agrees with the plant input files (12 large100 cases)",
       "PASS" if fp4.rel_err.max() < 1e-4 else "FAIL",
       f"{len(fp4)} (case,tech,year) points; max rel err {fp4.rel_err.max():.2e} at ({w.case}, {w.i}, {w.t})")
hr_bad = []
for c in LARGE_CASES:
    hr = load(c, "heat_rate")
    hr = hr[hr["i"].isin(NUC)].groupby("i")["Value"].agg(["min", "max"])
    for itech in hr.index:
        exp = float(plantchar(META[c]["pc"][itech])["heatrate"].iloc[-1])
        if not (abs(hr.loc[itech, "min"] - exp) < 0.05 and abs(hr.loc[itech, "max"] - exp) < 0.6):
            hr_bad.append((c, itech, hr.loc[itech].to_dict(), exp))
record("B3", "the heat rate data agrees with the plant input files", "PASS" if not hr_bad else "INFO",
       hr_bad[:3] if hr_bad else "both techs, all 12 cases (old vintages can differ a little)")
vom_bad = []
for c in LARGE_CASES:
    vom = load(c, "cost_vom")
    for itech in NUC:
        v = vom[vom["i"] == itech]
        if not len(v):
            continue
        got = float(v["Value"].mode().iloc[0])
        exp = float(plantchar(META[c]["pc"][itech])["vom"].iloc[-1]) * D2022
        if abs(got - exp) / exp > 0.02:
            vom_bad.append((c, itech, round(got, 3), round(exp, 3)))
record("B3", "the variable cost (VOM) data agrees with the plant input files", "PASS" if not vom_bad else "FAIL",
       vom_bad[:5] or "modal VOM within 2%, all 12 cases")
fin4, _ = fin_mult_decomposition(LARGE_CASES, {c: META[c]["fin"] for c in LARGE_CASES})
fin4.to_csv(S4 / "fin_mult_noITC_comparison.csv", index=False)
record("B3", "the finance multiplier has the correct regional structure",
       "PASS" if (fin4.ratio_spread_over_t < 2e-3).all() else "FAIL",
       f"max ratio spread over years {fin4.ratio_spread_over_t.max():.2e} (12 cases)")
record("B3", "the finance multiplier agrees with our own calculation",
       "PASS" if (fin4.worst_resid_after_regional < 1.5e-3).all() else "FAIL",
       f"worst |residual| {fin4.worst_resid_after_regional.max():.2e} (GAMS rounds to 3 decimals)")
ok_reg = (((fin4.reg_factor_mean - (1 + county.mean())).abs() < 0.02).all()
          and (fin4.reg_factor_min > lo_b).all() and (fin4.reg_factor_max < hi_b).all())
record("B3", "the regional factors are inside the county data range", "PASS" if ok_reg else "FAIL",
       f"implied range [{fin4.reg_factor_min.min():.4f}, {fin4.reg_factor_max.max():.4f}] "
       f"inside county range [{lo_b:.4f}, {hi_b:.4f}]")

[PASS] B3 :: the capital cost data agrees with the plant input files (12 large100 cases) -- 984 (case,tech,year) points; max rel err 2.25e-07 at (large100_eo_p05, nuclear, 2046)


[PASS] B3 :: the heat rate data agrees with the plant input files -- both techs, all 12 cases (old vintages can differ a little)


[PASS] B3 :: the variable cost (VOM) data agrees with the plant input files -- modal VOM within 2%, all 12 cases


[PASS] B3 :: the finance multiplier has the correct regional structure -- max ratio spread over years 7.41e-04 (12 cases)
[PASS] B3 :: the finance multiplier agrees with our own calculation -- worst |residual| 8.42e-04 (GAMS rounds to 3 decimals)
[PASS] B3 :: the regional factors are inside the county data range -- implied range [0.9529, 1.2539] inside county range [0.9456, 1.2717]


### Phases C to F: mandate mechanics, direction of the objective, load, cross-case gates

The shadow-price table and the binding-year table use the same definitions as the step-3 tables.
Phase D tests the direction of the objective change against the base run: cheaper inputs (`gaslo`,
`relo`) must not raise the system cost, dearer inputs and more load must not lower it. `translim`
changes constraints in a sequential (myopic) solve, so its shift has no pass condition.

In [14]:
TRAJ = {c: trajectory(META[c]["scen"]) for c in CASES}
DUAL, RAW = {}, {}
for c in CASES + sorted(set(BASE_CASES) | set(LARGE_P50)):
    DUAL[c], RAW[c] = dual_series(c)
MAND_CAP = {c: (lambda cap: cap[cap["i"] == META[c]["mandated_tech"]].groupby("t")["Value"].sum())(load(c, "cap"))
            for c in CASES}
duals4, viol, bind_bad, slack_bad = duals_table(CASES, YEARS_RUN, TRAJ, MAND_CAP, DUAL, RAW)
duals4.to_csv(S4 / "duals_by_year.csv", index=False)
assert len(duals4) == 120 * 16, "120 runs x 16 model years"
record("C", "the capacity satisfies the mandate floor in every case", "PASS" if not viol else "FAIL",
       viol[:5] or "no violation larger than 5 MW in any mandated year")
record("C", "the capacity equals the floor in each binding year (positive dual)", "PASS" if not bind_bad else "FAIL",
       bind_bad[:5] or "every year with dual > $1/MW-yr has |slack| <= 5 MW")
record("C", "a positive dual occurs only in the binding years", "PASS" if not slack_bad else "FAIL",
       slack_bad[:5] or "every year with slack > 100 MW has a zero dual")
gate_dual_conversion(record, "C", CASES, DUAL, RAW)
bad_large = [(c, float(v)) for c in SENS_CASES
             if (v := load(c, "cap_new_ann").query("i == 'nuclear' and t > 2030")["Value"].sum()) > 0]
bad_smr = [(c, float(v)) for c in LARGE_CASES
           if (v := load(c, "cap_new_ann").query("i == 'nuclear-smr'")["Value"].sum()) > 0]
record("C", "no large-reactor builds occur after 2030 in the sensitivity cases", "PASS" if not bad_large else "FAIL",
       bad_large or "all 108 sensitivity cases clean")
record("C", "no SMR builds occur in the large100 percentile cases", "PASS" if not bad_smr else "FAIL",
       bad_smr or "all 12 large100 cases clean")
pre = {c: float(load(c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in SENS_CASES}
record("C", "no SMR capacity exists before 2031", "PASS" if max(pre.values()) == 0 else "FAIL",
       f"max pre-2031 SMR capacity {max(pre.values()):.1f} MW")
ob4 = overbuild_table(duals4, CASES, YEARS_RUN, TRAJ)
ob4.to_csv(S4 / "overbuild_by_case.csv", index=False)
unbound = ob4[ob4.n_binding_years == 0]
record("C", "overbuild and binding years by case", "INFO",
       f"{len(unbound)} cases with zero binding years: {list(unbound.case)}" if len(unbound)
       else "every case has at least one binding year")

# ---- phase D: direction of the objective change -------------------------------------------
dir_bad, translim_rel = [], []
for c in SENS_CASES:
    b, s = META[c]["base"], META[c]["sens"]
    rel = (obj[c] - obj[b]) / obj[b]
    if (s in {"gashi", "rehi", "demhi"} and rel < -1e-6) or (s in {"gaslo", "relo"} and rel > 1e-6):
        dir_bad.append((c, round(rel, 6)))
    if s == "translim":
        translim_rel.append(rel)
record("D", "the objective moves in the expected direction against the base case", "PASS" if not dir_bad else "FAIL",
       dir_bad[:8] or "gaslo/relo never above base; gashi/rehi/demhi never below base")
record("D", "translim objective shift (no pass condition: myopic path dependence)", "INFO",
       f"range {min(translim_rel):+.4f} to {max(translim_rel):+.4f} rel to base")
record("D", "mean objective shift by sensitivity arm", "INFO",
       {s: round(float(np.mean([(obj[c] - obj[META[c]['base']]) / obj[META[c]['base']]
                                for c in SENS_CASES if META[c]["sens"] == s])), 4) for s in SENS})
br_bad = [(tok, o) for tok in SCHEDULES
          if not ((o := [obj[f"large100_{tok}_{p}"] for p in ["p05", "p50", "p95"]])[0] <= o[1] <= o[2])]
record("D", "the large100 percentile objectives bracket the Step 3 p50 objective", "PASS" if not br_bad else "FAIL",
       br_bad or "p05 <= p50 <= p95 for all 6 schedules")

# ---- phase E: load --------------------------------------------------------------------------
li4 = load_invariance([c for c in SENS_CASES + LARGE_CASES if META[c]["sens"] != "demhi"],
                      {c: (META[c]["base"] if META[c]["sens"] else REF) for c in CASES})
li4.to_csv(S4 / "load_invariance.csv", index=False)
record("E", "the exogenous load is identical to the base case outside the demhi arm",
       "PASS" if li4.max_rel_diff.max() < 1e-6 else "FAIL",
       f"{len(li4)} cases; max rel difference {li4.max_rel_diff.max():.2e}")
gate_dropped_load(record, "E", CASES)
record("E", "the high-electrification profile file is complete and correct", "INFO",
       "dropped in the port: an integrity scan of an HDF5 input profile, not of a run output; "
       "B2 tests the demhi load the runs used", port="moved")

# ---- phase F --------------------------------------------------------------------------------
mono_bad = []
for tok, s in [(tok, s) for tok in SCHEDULES for s in SENS] + [(tok, "large") for tok in SCHEDULES]:
    names = ([f"large100_{tok}_{p}" for p in ["p05", "p50", "p95"]] if s == "large"
             else [f"smr100_{tok}_{p}_{s}" for p in ["p05", "p50", "p95"]])
    d05, d50, d95 = (DUAL.get(n, pd.Series(dtype=float)) for n in names)
    idx = sorted(set(d05.index) | set(d50.index) | set(d95.index))
    if not idx:
        continue
    d05, d50, d95 = (x.reindex(idx).fillna(0.0) for x in (d05, d50, d95))
    tol = 50.0 + 0.001 * d95.abs()
    bad_years = [int(t) for t in idx if d95[t] < d50[t] - tol[t] or d50[t] < d05[t] - tol[t]]
    if bad_years:
        mono_bad.append((tok, s, bad_years))
record("F", "the dual values do not cross between p05, p50, and p95 (42 cells)", "PASS" if not mono_bad else "FAIL",
       mono_bad or "p95 >= p50 >= p05 in every year of every (schedule, sensitivity) cell and every large100 schedule")
ord_bad = [(tok, s, o) for tok in SCHEDULES for s in SENS
           if not ((o := [obj[f"smr100_{tok}_{p}_{s}"] for p in ["p05", "p50", "p95"]])[0] <= o[1] <= o[2])]
record("F", "the objective values are in the correct order (p05 <= p50 <= p95)", "PASS" if not ord_bad else "FAIL",
       ord_bad or "a more expensive world always costs more, all 36 cells")
gate_data_scan(record, "F", CASES)
gate_national_totals(record, "F", CASES)
decay_by_arm = {}
for s in SENS:
    d_ = {}
    for tok in SCHEDULES:
        for p in ["p05", "p50", "p95"]:
            dd = DUAL[f"smr100_{tok}_{p}_{s}"]
            dd = dd[dd > 1.0].sort_index()
            if len(dd) >= 2:
                d_[f"{tok}_{p}"] = round(float(dd.iloc[-1] / dd.max()), 2)
    decay_by_arm[s] = d_
record("F", "dual end/peak ratio by case and sensitivity arm (bridge, first look)", "INFO", decay_by_arm)
l_vs_s = {}
for tok in SCHEDULES:
    for p in ["p05", "p95"]:
        dl_ = DUAL[f"large100_{tok}_{p}"]
        ds_ = dual_series(f"smr100_{tok}_{p}")[0]
        shared = sorted(set(dl_[dl_ > 1.0].index) & set(ds_[ds_ > 1.0].index))
        if shared:
            l_vs_s[f"{tok}_{p}"] = round(float(dl_.reindex(shared).mean() / ds_.reindex(shared).mean()), 2)
record("F", "large100 dual level over smr100 dual level (shared binding years, p05/p95)", "INFO",
       f"mean ratio: {l_vs_s}")

summary4 = write_registry("step4_checks")

[PASS] C :: the capacity satisfies the mandate floor in every case -- no violation larger than 5 MW in any mandated year
[PASS] C :: the capacity equals the floor in each binding year (positive dual) -- every year with dual > $1/MW-yr has |slack| <= 5 MW
[PASS] C :: a positive dual occurs only in the binding years -- every year with slack > 100 MW has a zero dual
[PASS] C :: the dual conversion is correct (raw = cost_scale x pvf_onm x converted) -- max rel err 1.29e-07 across all cases with duals


[PASS] C :: no large-reactor builds occur after 2030 in the sensitivity cases -- all 108 sensitivity cases clean
[PASS] C :: no SMR builds occur in the large100 percentile cases -- all 12 large100 cases clean


[PASS] C :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW


[INFO] C :: overbuild and binding years by case -- every case has at least one binding year
[PASS] D :: the objective moves in the expected direction against the base case -- gaslo/relo never above base; gashi/rehi/demhi never below base
[INFO] D :: translim objective shift (no pass condition: myopic path dependence) -- range -0.0103 to +0.0015 rel to base
[INFO] D :: mean objective shift by sensitivity arm -- {'gaslo': -0.1437, 'gashi': 0.1055, 'demhi': 0.2219, 'relo': -0.0833, 'rehi': 0.0876, 'translim': -0.0059}
[PASS] D :: the large100 percentile objectives bracket the Step 3 p50 objective -- p05 <= p50 <= p95 for all 6 schedules


[PASS] E :: the exogenous load is identical to the base case outside the demhi arm -- 102 cases; max rel difference 0.00e+00


[PASS] E :: no load is dropped after 2025 -- all 120 cases clean; historical 2010-2023 artifact 2.91 TWh, case spread 333022.5 MWh
[INFO] E :: the high-electrification profile file is complete and correct -- dropped in the port: an integrity scan of an HDF5 input profile, not of a run output; B2 tests the demhi load the runs used
[PASS] F :: the dual values do not cross between p05, p50, and p95 (42 cells) -- p95 >= p50 >= p05 in every year of every (schedule, sensitivity) cell and every large100 schedule
[PASS] F :: the objective values are in the correct order (p05 <= p50 <= p95) -- a more expensive world always costs more, all 36 cells


[PASS] F :: no data value is NaN or infinite in any file -- all 209 keys x 120 files scanned (key_inventory.n_nonfinite)
[PASS] F :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (key_inventory.min_*)
[PASS] F :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 120 cases (storage charging and upgrades make some non-nuclear rows negative; a normal convention)


[PASS] F :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 120 cases within 1e-3
[INFO] F :: dual end/peak ratio by case and sensitivity arm (bridge, first look) -- {'gaslo': {'eia_p05': 0.21, 'eia_p50': 0.75, 'eia_p95': 0.82, 'aj_p05': 0.26, 'aj_p50': 0.81, 'aj_p95': 0.8, 'iaea_p05': 0.31, 'iaea_p50': 0.35, 'iaea_p95': 0.83, 'mck_p05': 0.27, 'mck_p50': 0.52, 'mck_p95': 0.8, 'cop28_p05': 0.29, 'cop28_p50': 0.46, 'cop28_p95': 0.71, 'eo_p05': 0.37, 'eo_p50': 0.84, 'eo_p95': 0.72}, 'gashi': {'eia_p50': 0.7, 'eia_p95': 0.79, 'aj_p05': 0.09, 'aj_p50': 0.55, 'aj_p95': 0.67, 'iaea_p05': 0.13, 'iaea_p50': 0.13, 'iaea_p95': 0.72, 'mck_p05': 0.17, 'mck_p50': 0.25, 'mck_p95': 0.68, 'cop28_p05': 0.12, 'cop28_p50': 0.24, 'cop28_p95': 0.59, 'eo_p05': 0.11, 'eo_p50': 0.83, 'eo_p95': 0.69}, 'demhi': {'eia_p50': 0.69, 'eia_p95': 0.81, 'aj_p05': 0.09, 'aj_p50': 0.79, 'aj_p95': 0.8, 'iaea_p05': 0.1, 'iaea_p50': 0.27, 'iaea_p95': 0.78, 'mck_p05': 0.02, 'm

## 4. The itcfb batch (`itcfb_checks`): 19 credit-feedback runs

These runs feed the required subsidy back into ReEDS as a real investment tax credit (ITC) on the
SMR. The fed rate at each *rate year* (a year in which the base run's shadow price binds) is the
paper's headline required ITC rate plus a 0.01 indifference bump; ReEDS monetizes it at 0.9 (the
OBBBA 10 % tax-equity penalty) and applies it inside the financing multiplier. Five arms:
**fbA** (mandate + ITC; the shadow price must collapse), **fbB** (ITC only; the national SMR
capacity must reproduce the target trajectory), **fbC** (ITC + endogenous learning; the
pre-registered section-9 test), **fbK** (mandate + learning, no ITC; the learning engine must
reproduce the drawn cost path) and **fbR** (flat ITC at 0, 30 and 50 %).

The **recovered rate** `m` comes from the two financing multipliers ReEDS reports, with and without
the credit: `m = (1 - ratio) (1 - tax PVdep) / (1 - tax PVdep / 2)`, `ratio = fin_mult /
fin_mult_noITC`, `PVdep` the present value of depreciation. The construction, risk, evaluation,
degradation and regional factors cancel in the ratio. GAMS loads the multipliers at three decimals,
so the tolerance is |dm| <= 0.005.

**Delivery tiers** (national cumulative SMR capacity against the target trajectory): *exact* (every
gap within 5 MW), *reproduces* (every rate-year and 2050 shortfall within max(5 % of the target,
500 MW); overbuild never fails), *partial* (2050 shortfall within 15 %), *no* (above 15 %).

Derived tables and readers (all read by notebook 09): `duals_by_year.csv` (cap_MW is the national
SMR capacity), `trajectory_gaps.csv`, `rate_recovery.csv`, `fbK_calibration.csv`, `fbA_collapse.csv`,
`incentives_echo.csv` (the fed rates), `checks_summary.csv` (the QA gate).

In [15]:
record = registry("itcfb_checks")
SF = out_dir("itcfb_checks")

cases_fb = pd.read_csv(paths.casefile("itcfb"), index_col=0)
CASES = [c for c in cases_fb.columns if c != "Default Value"]
assert len(CASES) == 19, f"the itcfb casefile has 19 runs, got {len(CASES)}"
sw = lambda c, n: switch(cases_fb, c, n)
META = {}
for c in CASES:
    arm = c.split("_")[0]
    META[c] = dict(arm=arm, world=c.split("_")[1] if arm != "fbR" else "eia",
                   scen=sw(c, "GSw_NuclearCapMandateScen"), mandate=int(float(sw(c, "GSw_NuclearCapMandate"))),
                   learning=int(float(sw(c, "GSw_NuclearLearning"))), suffix=sw(c, "incentives_suffix"),
                   foreign=sw(c, "GSw_NuclearLearning_ForeignScen"),
                   pc={"nuclear": sw(c, "plantchar_nuclear"), "nuclear-smr": sw(c, "plantchar_nuclear_smr")},
                   fin=sw(c, "financials_tech_suffix"))
ARM_CASES = {a: [c for c in CASES if META[c]["arm"] == a] for a in ["fbA", "fbB", "fbC", "fbK", "fbR"]}
assert [len(ARM_CASES[a]) for a in ["fbA", "fbB", "fbC", "fbK", "fbR"]] == [2, 6, 6, 2, 3]
FED_CASES = ARM_CASES["fbA"] + ARM_CASES["fbB"] + ARM_CASES["fbC"]
NOITC_CASES = ARM_CASES["fbK"] + ["fbR_none"]
LEARNOFF_CASES = ARM_CASES["fbA"] + ARM_CASES["fbB"] + ARM_CASES["fbR"]
MANDATE_CASES = ARM_CASES["fbA"] + ARM_CASES["fbK"]
BASE_OF_WORLD = {w: f"smr100_{w}_p50" for w in SCHEDULES}
BASE_CASES = sorted(BASE_OF_WORLD.values())
REF4 = "smr100_eia_p50_gaslo"

# ---- fed rates: read from the incentives files the runs used (see section 1) -------------
EPS, PEN = 0.01, 0.1
RATES, M_EXP = {}, {}
for w in SCHEDULES:
    inc = pd.read_csv(FIN_DIR / f"incentives_obbba_itcfb_{w}_p50.csv")
    n = inc[inc["i"] == "Nuclear-SMR"]
    RATES[w] = {int(t): float(f) for t, f in zip(n["t_start_construction"], n["itc_frac"])}
    M_EXP[w] = {t: f * (1.0 - PEN) for t, f in RATES[w].items()}
FLAT_M = {"fbR_none": 0.0, "fbR_itc30": 0.30 * (1 - PEN), "fbR_itc50": 0.50 * (1 - PEN)}
print("rate years per world:", {w: sorted(RATES[w]) for w in SCHEDULES})

# ---- phase A ----------------------------------------------------------------------------------
gate_census(record, "A", "itcfb", CASES, "one output file exists for each of the 19 cases")
gate_census(record, "A", "smr100", BASE_CASES, "the six Step 3 base-case files are available for comparison")
record("A", "the folder census matches the consolidated delivery", "INFO",
       f"{int((INV.batch == 'itcfb').sum())} itcfb runs in the key inventory; archive layout per batch "
       "replaces the consolidated delivery folder", port="adapted")
keysets, union = gate_keys(record, "A", CASES,
                           "all 19 files have the same data keys (dual keys can be absent when zero)",
                           ref=REF4, ref_check="the key set agrees with the Step 4 vintage")
no_dual = sorted(c for c in CASES if "nuclear_cap_price" not in KEYS[c])
record("A", "cases with no dual key (all-zero dual)", "INFO",
       f"{no_dual} -- expected: the 15 no-mandate cases, plus fbA if the credit worked")
EXPECT_YEARS = [int(y) for y in sw(CASES[0], "yearset").split("_")]
SOLVE_YEARS = [t for t in EXPECT_YEARS if t >= 2026]
assert len(SOLVE_YEARS) == 12, SOLVE_YEARS
YEARS_RUN = EXPECT_YEARS
obj = gate_solve_health(record, "A", CASES, EXPECT_YEARS)

rate years per world: {'eia': [2038, 2041, 2044, 2047, 2050], 'aj': [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050], 'iaea': [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050], 'mck': [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050], 'cop28': [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050], 'eo': [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]}
[PASS] A :: one output file exists for each of the 19 cases -- 19 runs returned (key_inventory: batch itcfb)
[PASS] A :: the six Step 3 base-case files are available for comparison -- 6 runs returned (key_inventory: batch smr100)
[INFO] A :: the folder census matches the consolidated delivery -- 19 itcfb runs in the key inventory; archive layout per batch replaces the consolidated delivery folder
[PASS] A :: all 19 files have the same data keys (dual keys can be absent when zero) -- key counts [207, 209]; runs without a shadow-price table: 17
[PASS] A :: the key set agrees with the Ste

[PASS] A :: the objective value is a normal positive number in every file -- range 1.3854e+12 to 1.6378e+12
[PASS] A :: the model years are correct in every file (annual 2031-2035 block present) -- expected [2010, 2015, 2020, 2023, 2026, 2029, 2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]
[PASS] A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 at 14.7645 (max dev 9.5e-07); cost_scale==1; z_rep has 16 years in every file


In [16]:
# ---- B1: design matrix ------------------------------------------------------------------------
EXP_DESIGN = {"fbA": (1, 1, 0), "fbB": (0, 1, 0), "fbC": (0, 1, 1), "fbK": (1, 0, 1), "fbR": (0, None, 0)}
mx_bad = []
for c in CASES:
    m_ = META[c]
    fed = m_["suffix"].startswith("obbba_itcfb_")
    exp_m, exp_f, exp_l = EXP_DESIGN[m_["arm"]]
    ok = m_["mandate"] == exp_m and m_["learning"] == exp_l and (exp_f is None or fed == bool(exp_f))
    if m_["arm"] == "fbR":
        ok = ok and m_["suffix"] == {"fbR_none": "obbba_nonuclearitc", "fbR_itc30": "obbba_itcflat30",
                                     "fbR_itc50": "obbba_itcflat50"}[c]
    if fed:
        ok = ok and m_["suffix"] == f"obbba_itcfb_{m_['world']}_p50"
    if m_["learning"]:
        ok = ok and m_["foreign"] == f"fb_{m_['world']}_p50"
    if not ok:
        mx_bad.append(c)
record("B", "the case matrix encodes the intended mandate, ITC, and learning design",
       "PASS" if not mx_bad else "FAIL", mx_bad or "19 cases match the three-factor design and the naming rules")

# ---- B2: fed incentives files (conventions; the t09 equality moves to notebook 09) --------------
echo_rows, b2_bad = [], []
for w in SCHEDULES:
    inc = pd.read_csv(FIN_DIR / f"incentives_obbba_itcfb_{w}_p50.csv")
    nuc_rows = inc[inc["i"].str.lower().str.startswith("nuclear")]
    only_smr = set(nuc_rows["i"]) == {"Nuclear-SMR"}
    conv_ok = bool((nuc_rows["safe_harbor"] == 0).all()
                   and (nuc_rows["t_max_online"] == nuc_rows["t_start_construction"]).all()
                   and (nuc_rows["itc_tax_equity_penalty"] == PEN).all()
                   and (nuc_rows["itc_energy_comm_bonus"] == 0.0).all()
                   and (nuc_rows["itc_percpt_domestic_bonus"] == 0.0).all())
    if not (only_smr and conv_ok):
        b2_bad.append((w, dict(only_smr=only_smr, conv_ok=conv_ok)))
    for t, f in sorted(RATES[w].items()):
        echo_rows.append(dict(world=w, t=t, itc_frac_file=f))
pd.DataFrame(echo_rows).to_csv(SF / "incentives_echo.csv", index=False)
record("B", "the fed incentives files follow the credit conventions at the rate years", "PASS" if not b2_bad else "FAIL",
       b2_bad or "6 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0", port="adapted")
record("B", "the fed incentives files equal the headline rate schedule plus the bump at exactly the rate years",
       "INFO", "moved to notebook 09: it needs t09_required_itc.csv from notebook 07 (itc_frac == "
       "round(i_model_headline + 0.01, 3))", port="moved")
b3_bad = []
for name, frac in [("obbba_itcflat30", 0.30), ("obbba_itcflat50", 0.50)]:
    inc = pd.read_csv(FIN_DIR / f"incentives_{name}.csv")
    nuc_rows = inc[inc["i"].str.lower().str.startswith("nuclear")]
    for tech in ["Nuclear", "Nuclear-SMR"]:
        sub = nuc_rows[nuc_rows["i"] == tech]
        yrs = sorted(sub["t_start_construction"].astype(int))
        if yrs != SOLVE_YEARS or not (sub["itc_frac"] == frac).all():
            b3_bad.append((name, tech, yrs[:3]))
    if set(nuc_rows["i"]) != {"Nuclear", "Nuclear-SMR"}:
        b3_bad.append((name, "techs", sorted(set(nuc_rows["i"]))))
record("B", "the flat incentives files credit both nuclear technologies at 0.30 and 0.50 in all 12 solve years",
       "PASS" if not b3_bad else "FAIL", b3_bad or "both files correct, 2026-2050")

[PASS] B :: the case matrix encodes the intended mandate, ITC, and learning design -- 19 cases match the three-factor design and the naming rules
[PASS] B :: the fed incentives files follow the credit conventions at the rate years -- 6 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0
[INFO] B :: the fed incentives files equal the headline rate schedule plus the bump at exactly the rate years -- moved to notebook 09: it needs t09_required_itc.csv from notebook 07 (itc_frac == round(i_model_headline + 0.01, 3))
[PASS] B :: the flat incentives files credit both nuclear technologies at 0.30 and 0.50 in all 12 solve years -- both files correct, 2026-2050


In [17]:
# ---- rate recovery ----------------------------------------------------------------------------
rr_rows = []
for c in CASES:
    g = recovered_m(c, META[c]["fin"]).groupby(["i", "t"])["m_rec"].agg(["median", "min", "max"])
    g["spread_r"] = g["max"] - g["min"]
    w = META[c]["world"]
    for (i, t), r in g.iterrows():
        if META[c]["arm"] == "fbR":
            m_exp = FLAT_M[c] if (t in SOLVE_YEARS and c != "fbR_none") else 0.0
        else:
            m_exp = (M_EXP[w].get(t, 0.0) if (i == "nuclear-smr" and META[c]["suffix"].startswith("obbba_itcfb_"))
                     else 0.0)
        rr_rows.append(dict(case=c, arm=META[c]["arm"], i=i, t=t, m_expected=round(m_exp, 4),
                            m_recovered=round(float(r["median"]), 4), spread_r=round(float(r["spread_r"]), 5),
                            delta=round(float(r["median"]) - m_exp, 4)))
rr = pd.DataFrame(rr_rows)
rr.to_csv(SF / "rate_recovery.csv", index=False)
TOL_M = 0.005

def rate_year_rows(case_list):
    sub = rr[rr["case"].isin(case_list) & (rr["i"] == "nuclear-smr")]
    return sub[[t in RATES[META[c]["world"]] for c, t in zip(sub["case"], sub["t"])]]

b4 = rate_year_rows(ARM_CASES["fbA"] + ARM_CASES["fbB"])
w4 = b4.loc[b4["delta"].abs().idxmax()]
record("B", "the recovered ITC rate equals the fed rate at exactly the rate years (fbA and fbB)",
       "PASS" if b4["delta"].abs().max() <= TOL_M else "FAIL",
       f"{len(b4)} rate case-years; worst |dm| {b4['delta'].abs().max():.4f} at ({w4.case}, {w4.t})")
b4c = rate_year_rows(ARM_CASES["fbC"])
w4c = b4c.loc[b4c["delta"].abs().idxmax()]
record("B", "the recovered ITC rate equals the fed rate in the learning runs (fbC)",
       "PASS" if b4c["delta"].abs().max() <= TOL_M else "INFO",
       f"{len(b4c)} rate case-years; worst |dm| {b4c['delta'].abs().max():.4f} at ({w4c.case}, {w4c.t})")
fed = rr[rr["case"].isin(FED_CASES)]
off = fed[[not (i == "nuclear-smr" and t in RATES[META[c]["world"]]) for c, i, t in zip(fed["case"], fed["i"], fed["t"])]]
record("B", "the recovered rate is zero outside the rate years and for the large reactor in the fed arms",
       "PASS" if off["m_recovered"].abs().max() <= TOL_M else "FAIL",
       f"{len(off)} points; max |m| {off['m_recovered'].abs().max():.4f}")
b6_bad = []
for c in ["fbR_itc30", "fbR_itc50"]:
    sub = rr[(rr["case"] == c) & rr["t"].isin(SOLVE_YEARS)]
    for i in NUC:
        expect_years = [t for t in SOLVE_YEARS if t >= 2031] if i == "nuclear-smr" else SOLVE_YEARS
        got = sub[sub["i"] == i].set_index("t")["m_recovered"]
        miss = [t for t in expect_years if t not in got.index]
        badv = [(t, v) for t, v in got.items() if abs(v - FLAT_M[c]) > TOL_M]
        if miss or badv:
            b6_bad.append((c, i, miss, badv[:3]))
record("B", "the flat arms recover 0.27 and 0.45 for both nuclear technologies in every covered solve year",
       "PASS" if not b6_bad else "FAIL", b6_bad or "12 large + 10 SMR (from 2031) solve years per case")
worst_wedge, wedge_at = 0.0, None
for c in NOITC_CASES:
    m = float((nuc_slice(c, "cost_cap_fin_mult") - nuc_slice(c, "cost_cap_fin_mult_noITC")).abs().max())
    if m > worst_wedge:
        worst_wedge, wedge_at = m, c
record("B", "no ITC reaches nuclear technology in fbK and fbR_none (fin_mult == fin_mult_noITC)",
       "PASS" if worst_wedge < 1e-6 else "FAIL",
       f"max |difference| {worst_wedge:.2e}" + (f" at {wedge_at}" if worst_wedge >= 1e-6 else ""))
sp = rr[rr["case"].isin(FED_CASES) & (rr["i"] == "nuclear-smr")]
record("B", "the recovered rate is uniform across regions", "PASS" if sp["spread_r"].max() <= TOL_M else "FAIL",
       f"max spread over regions {sp['spread_r'].max():.5f} (14 fed cases)")
# Without the phaseout exemption, GSw_TCPhaseout_forceyear=2032 would cut every credit from online year 2036 to 0.
late = rr[(rr["t"] >= 2036) & (rr["m_expected"] > 0)]
record("B", "the phaseout does not reduce the nuclear credit in any year (exemption held)",
       "PASS" if late["delta"].abs().max() <= TOL_M else "FAIL",
       f"{len(late)} credited case-years at t >= 2036; worst |dm| {late['delta'].abs().max():.4f}")
leak_rows = itc_wedge_leak([("fbB_eia_p50", "smr100_eia_p50"), ("fbC_eo_p50", "smr100_eo_p50"),
                            ("fbR_itc50", "smr100_eia_p50")])
record("B", "the phaseout exemption did not change the other technologies (non-nuclear ITC wedge identical to the base run)",
       "PASS" if not leak_rows else "FAIL", leak_rows or "3 spot cases vs their base runs; wedge identical")
pay_bad = []
for c in CASES:
    sc = load(c, "systemcost_techba")
    v = float(sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]["Value"].abs().sum())
    has_itc = c in FED_CASES or c in ("fbR_itc30", "fbR_itc50")
    if has_itc and v < 1e6:
        pay_bad.append((c, "expected nuclear ITC payments, found none"))
    if not has_itc and v > 1.0:
        pay_bad.append((c, f"unexpected nuclear ITC payments {v:,.0f}"))
record("B", "the system cost data shows nuclear ITC payments only in the ITC arms", "PASS" if not pay_bad else "FAIL",
       pay_bad or "payments present in the 16 ITC cases, absent in fbK and fbR_none")
itc_mag = {c: abs(float(load(c, "tax_expenditure_itc")["Value"].sum())) for c in CASES}
record("B", "the other technologies keep their ITC in all 19 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
       f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

[PASS] B :: the recovered ITC rate equals the fed rate at exactly the rate years (fbA and fbB) -- 70 rate case-years; worst |dm| 0.0001 at (fbB_aj_p50, 2031)
[PASS] B :: the recovered ITC rate equals the fed rate in the learning runs (fbC) -- 55 rate case-years; worst |dm| 0.0001 at (fbC_aj_p50, 2031)
[PASS] B :: the recovered rate is zero outside the rate years and for the large reactor in the fed arms -- 211 points; max |m| 0.0000
[PASS] B :: the flat arms recover 0.27 and 0.45 for both nuclear technologies in every covered solve year -- 12 large + 10 SMR (from 2031) solve years per case
[PASS] B :: no ITC reaches nuclear technology in fbK and fbR_none (fin_mult == fin_mult_noITC) -- max |difference| 0.00e+00
[PASS] B :: the recovered rate is uniform across regions -- max spread over regions 0.00090 (14 fed cases)
[PASS] B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 90 credited case-years at t >= 2036; worst |dm| 0.0001


[PASS] B :: the phaseout exemption did not change the other technologies (non-nuclear ITC wedge identical to the base run) -- 3 spot cases vs their base runs; wedge identical


[PASS] B :: the system cost data shows nuclear ITC payments only in the ITC arms -- payments present in the 16 ITC cases, absent in fbK and fbR_none
[PASS] B :: the other technologies keep their ITC in all 19 runs -- total ITC tax expenditure magnitude 123.8 to 1227.3 B$ per case


In [18]:
fpf = cost_fingerprint(CASES, {c: META[c]["pc"] for c in CASES}, {c: META[c]["learning"] for c in CASES})
fpf.to_csv(SF / "fingerprint_errors.csv", index=False)
off_fp = fpf[fpf["learning"] == 0]
w = off_fp.loc[off_fp.rel_err.idxmax()]
record("B", "the learning-off runs use the drawn cost path without change (11 cases)",
       "PASS" if off_fp.rel_err.max() < 1e-4 else "FAIL",
       f"{len(off_fp)} (case, tech, year) points; max rel err {off_fp.rel_err.max():.2e} at ({w.case}, {w.i}, {w.t})")
on_fp = fpf[(fpf["learning"] == 1) & (fpf["t"] >= 2031)]
record("B", "the learning-on runs write an in-run cost path (deviation from the drawn path is expected)", "INFO",
       on_fp.groupby(["case", "i"])["rel_err"].max().round(4).to_dict())
dual_present = {c: "nuclear_cap_price" in KEYS[c] for c in CASES}
b15_bad = [c for c in CASES if META[c]["mandate"] == 0 and dual_present[c]]
record("B", "the dual keys exist only where a mandate exists", "PASS" if not b15_bad else "FAIL",
       b15_bad or f"absent in all 15 no-mandate cases; fbA: {[dual_present[c] for c in ARM_CASES['fbA']]}; "
       f"fbK: {[dual_present[c] for c in ARM_CASES['fbK']]}")
b16_worst = {}
for w, group in [("eia", ["fbA_eia_p50", "fbB_eia_p50", "fbR_none", "fbR_itc30", "fbR_itc50"]),
                 ("eo", ["fbA_eo_p50", "fbB_eo_p50"])]:
    for key in ["cost_cap", "cost_cap_fin_mult_noITC"]:
        a0 = nuc_slice(group[0], key)
        for c in group[1:]:
            b16_worst[(w, key)] = max(b16_worst.get((w, key), 0.0), float((nuc_slice(c, key) - a0).abs().max()))
record("B", "the no-ITC finance multiplier and the capital cost are identical across the learning-off arms of one world",
       "PASS" if max(b16_worst.values()) < 1e-6 else "FAIL",
       f"max |difference| {max(b16_worst.values()):.2e} over {sorted(set(k[0] for k in b16_worst))}")

[PASS] B :: the learning-off runs use the drawn cost path without change (11 cases) -- 902 (case, tech, year) points; max rel err 1.83e-07 at (fbB_iaea_p50, nuclear-smr, 2049)
[INFO] B :: the learning-on runs write an in-run cost path (deviation from the drawn path is expected) -- {('fbC_aj_p50', 'nuclear'): 0.0001, ('fbC_aj_p50', 'nuclear-smr'): 0.1046, ('fbC_cop28_p50', 'nuclear'): 0.003, ('fbC_cop28_p50', 'nuclear-smr'): 0.2702, ('fbC_eia_p50', 'nuclear'): 0.0013, ('fbC_eia_p50', 'nuclear-smr'): 0.1882, ('fbC_eo_p50', 'nuclear'): 0.0482, ('fbC_eo_p50', 'nuclear-smr'): 0.0883, ('fbC_iaea_p50', 'nuclear'): 0.1734, ('fbC_iaea_p50', 'nuclear-smr'): 0.3501, ('fbC_mck_p50', 'nuclear'): 0.0053, ('fbC_mck_p50', 'nuclear-smr'): 0.2387, ('fbK_eia_p50', 'nuclear'): 0.0011, ('fbK_eia_p50', 'nuclear-smr'): 0.1882, ('fbK_eo_p50', 'nuclear'): 0.0187, ('fbK_eo_p50', 'nuclear-smr'): 0.0307}
[PASS] B :: the dual keys exist only where a mandate exists -- absent in all 15 no-mandate cases; fbA: [False,

### Phase C: the pre-registered criteria, arm by arm

The control order matters: fbA first (a persisting shadow price means the credit was misapplied),
then fbB and fbK, then fbC. `trajectory_gaps.csv` holds the gap between the national SMR capacity
and the target in every mandated year of the fed and mandated runs; `status` is `rate` in a rate
year and `slack` otherwise (slack years get no credit, so their shortfalls are reported, not gated).
`fbA_collapse.csv` lists every binding base-run year beside the fbA shadow price; the base-run
prices come from this notebook's `step3_checks/duals_by_year.csv`. `fbK_calibration.csv` compares
the in-run learned capital cost with the drawn cost path under the engine's timing: the engine prices
solve year t from the experience through the previous solve year (`nuclear_learning.py`, `tprev`),
so at an override year (a solve year from 2031) the applied cost must sit between the drawn cost at t
and at the previous solve year.

In [19]:
TRAJ_W = {w: trajectory(f) for w, f in [("eia", "eia_aeo_high_smr"), ("aj", "abou_jaoude_smr"),
                                       ("iaea", "iaea_high_smr"), ("mck", "mckinsey_smr"),
                                       ("cop28", "cop28_smr"), ("eo", "eo2025_smr")]}
SCEN_OF_W = {"eia": "eia_aeo_high_smr", "aj": "abou_jaoude_smr", "iaea": "iaea_high_smr",
             "mck": "mckinsey_smr", "cop28": "cop28_smr", "eo": "eo2025_smr"}
assert all(META[c]["scen"] == SCEN_OF_W[META[c]["world"]] for c in CASES if META[c]["mandate"])
SMR_CAP = {c: load(c, "cap").query("i == 'nuclear-smr'").groupby("t")["Value"].sum() for c in CASES}
LARGE_NEW = {c: load(c, "cap_new_ann").query("i == 'nuclear' and t > 2030").groupby("t")["Value"].sum() for c in CASES}
DUAL, RAW = {}, {}
for c in CASES:
    DUAL[c], RAW[c] = dual_series(c)

gap_rows = []
for c in ARM_CASES["fbA"] + ARM_CASES["fbB"] + ARM_CASES["fbC"] + ARM_CASES["fbK"]:
    w = META[c]["world"]
    for t in YEARS_RUN:
        req = float(TRAJ_W[w].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP[c].get(t, 0.0))
        gap_rows.append(dict(case=c, arm=META[c]["arm"], world=w, t=t,
                             status="rate" if t in RATES[w] else "slack", mandate_MW=req,
                             cap_MW=round(have, 1), gap_MW=round(have - req, 1),
                             gap_pct=round(100 * (have - req) / req, 2)))
gaps = pd.DataFrame(gap_rows)
gaps.to_csv(SF / "trajectory_gaps.csv", index=False)
TIER = {c: gap_tier(gaps, c) for c in gaps["case"].unique()}
print(pd.Series(TIER).to_string())

duals3 = pd.read_csv(S3 / "duals_by_year.csv")     # the canonical base-run shadow prices (section 2)
c1_rows, c1_bad = [], []
for c in ARM_CASES["fbA"]:
    base = duals3[(duals3["case"] == BASE_OF_WORLD[META[c]["world"]]) & (duals3["dual_2004_MWyr"] > 1.0)]
    for t in base["t"]:
        d_fb = float(DUAL[c].get(t, 0.0))
        c1_rows.append(dict(case=c, t=int(t), base_dual_2004_MWyr=float(base.set_index("t").loc[t, "dual_2004_MWyr"]),
                            fbA_dual_2004_MWyr=d_fb, collapsed=d_fb <= 1.0))
        if d_fb > 1.0:
            c1_bad.append((c, int(t), round(d_fb, 1)))
c1 = pd.DataFrame(c1_rows)
c1.to_csv(SF / "fbA_collapse.csv", index=False)
record("C", "every previously binding floor dual collapses in the fbA runs", "PASS" if not c1_bad else "FAIL",
       c1_bad or f"{len(c1)} previously binding case-years; every fbA dual <= $1/MW-yr")
c2_bad = [(c, v[["t", "gap_MW"]].values.tolist()[:3]) for c in MANDATE_CASES
          if len(v := gaps[(gaps["case"] == c) & (gaps["gap_MW"] < -5.0)])]
record("C", "the capacity satisfies the mandate floor in the fbA and fbK runs", "PASS" if not c2_bad else "FAIL",
       c2_bad or "no shortfall larger than 5 MW in any mandated year (4 cases)")
record("C", "the fbA and fbK overbuild relative to the trajectory", "INFO",
       {c: round(float((gaps[gaps['case'] == c]['cap_MW'] / gaps[gaps['case'] == c]['mandate_MW']).max()), 3)
        for c in MANDATE_CASES})
fbB_tiers = {c: TIER[c] for c in ARM_CASES["fbB"]}
worst_b = gaps[gaps["case"].isin(ARM_CASES["fbB"]) & (gaps["status"] == "rate")]
record("C", "the national SMR capacity reproduces the mandate trajectory in the fbB runs",
       "PASS" if all(t in ("exact", "reproduces") for t in fbB_tiers.values()) else "FAIL",
       f"tiers: {fbB_tiers}; worst rate-year shortfall {float((-worst_b['gap_MW']).clip(lower=0).max()):,.0f} MW")
record("C", "the large-reactor substitution in the fbB runs (a report, not a failure)", "INFO",
       f"new large builds after 2030 (GW): { {c: round(float(LARGE_NEW[c].sum()) / 1000.0, 2) for c in ARM_CASES['fbB']} }")
record("C", "the fbC runs return complete data for the section-9 verdict", "INFO",
       f"tiers on the same convention: { {c: TIER[c] for c in ARM_CASES['fbC']} }; the verdict is read in notebook 09")

# ---- fbK calibration --------------------------------------------------------------------------
OVERRIDE_YEARS = [t for t in SOLVE_YEARS if t >= 2031]
TPREV = {t: EXPECT_YEARS[EXPECT_YEARS.index(t) - 1] for t in OVERRIDE_YEARS}
c7_rows = []
for c in ARM_CASES["fbK"]:
    cc = load(c, "cost_cap")
    cc = cc[cc["i"].isin(NUC)].set_index(["i", "t"])["Value"]
    for itech in NUC:
        drawn = plantchar(META[c]["pc"][itech])["capcost"] * 1000.0 * D2022
        for t, expect in drawn.items():
            if (itech, t) not in cc.index:
                continue
            got = float(cc.loc[(itech, t)])
            tol = 1e-3 * expect
            if t in TPREV:
                prev_v = float(drawn.get(TPREV[t], expect))
                lo, hi = min(expect, prev_v), max(expect, prev_v)   # the drawn path steps up at the 2030 anchor
                ok = (lo - tol) <= got <= (hi + tol)
            else:
                ok = abs(got - expect) <= 1e-6 * expect
            c7_rows.append(dict(case=c, i=itech, t=int(t), drawn_2004_MW=expect, inrun_2004_MW=got,
                                drawn_tprev_2004_MW=float(drawn.get(TPREV.get(t), expect)),
                                override=t in TPREV, in_bracket=ok, rel_err=abs(got - expect) / expect))
c7 = pd.DataFrame(c7_rows)
c7.to_csv(SF / "fbK_calibration.csv", index=False)
bad7 = c7[~c7["in_bracket"]]
record("C", "the in-run learned cost path reproduces the drawn cost path under the engine's lagged-experience timing (fbK)",
       "PASS" if not len(bad7) else "FAIL",
       bad7[["case", "i", "t"]].values.tolist()[:5] if len(bad7) else
       f"{int(c7['override'].sum())} override points inside [drawn(t), drawn(tprev)]; "
       f"{int((~c7['override']).sum())} non-override points echo the drawn input")
record("C", "the myopia lag between the applied cost and the same-year drawn cost (context)", "INFO",
       f"max same-year rel deviation by (case, tech): {c7[c7['override']].groupby(['case', 'i'])['rel_err'].max().round(4).to_dict()}")
c8 = {}
for c in ARM_CASES["fbK"]:
    base = duals3[(duals3["case"] == BASE_OF_WORLD[META[c]["world"]]) & (duals3["dual_2004_MWyr"] > 1.0)].set_index("t")["dual_2004_MWyr"]
    both = [t for t in base.index if float(DUAL[c].get(t, 0.0)) > 1.0]
    c8[c] = (f"{len(both)}/{len(base)} shared binding years, mean ratio "
             f"{np.mean([float(DUAL[c].get(t)) / float(base.loc[t]) for t in both]):.2f}" if both
             else f"0/{len(base)} shared binding years")
record("C", "the fbK duals stay close to the base-run duals (a replay, degenerate-tie sensitive)", "INFO", c8)
cap50 = {c: float(load(c, "cap").query("i in @NUC and t == 2050")["Value"].sum()) for c in ARM_CASES["fbR"]}
mono = cap50["fbR_none"] <= cap50["fbR_itc30"] + 5.0 and cap50["fbR_itc30"] <= cap50["fbR_itc50"] + 5.0
record("C", "the deployment increases with the flat credit rate in the fbR runs", "PASS" if mono else "FAIL",
       {c: f"{v/1000:.1f} GW nuclear in 2050" for c, v in cap50.items()})
r_none_ok = (not dual_present["fbR_none"]
             and float((nuc_slice("fbR_none", "cost_cap_fin_mult") - nuc_slice("fbR_none", "cost_cap_fin_mult_noITC")).abs().max()) < 1e-6
             and fpf[fpf["case"] == "fbR_none"].rel_err.max() < 1e-4)
record("C", "the fbR_none run is a clean no-subsidy baseline", "PASS" if r_none_ok else "FAIL",
       "no dual key; zero ITC wedge; drawn cost path unchanged")

fbA_eia_p50      reproduces
fbA_eo_p50       reproduces
fbB_eia_p50      reproduces
fbB_aj_p50       reproduces
fbB_iaea_p50     reproduces
fbB_mck_p50      reproduces
fbB_cop28_p50    reproduces
fbB_eo_p50       reproduces
fbC_eia_p50         partial
fbC_aj_p50       reproduces
fbC_iaea_p50     reproduces
fbC_mck_p50      reproduces
fbC_cop28_p50    reproduces
fbC_eo_p50       reproduces
fbK_eia_p50           exact
fbK_eo_p50            exact
[PASS] C :: every previously binding floor dual collapses in the fbA runs -- 15 previously binding case-years; every fbA dual <= $1/MW-yr
[PASS] C :: the capacity satisfies the mandate floor in the fbA and fbK runs -- no shortfall larger than 5 MW in any mandated year (4 cases)
[INFO] C :: the fbA and fbK overbuild relative to the trajectory -- {'fbA_eia_p50': 3.731, 'fbA_eo_p50': 2.297, 'fbK_eia_p50': 1.0, 'fbK_eo_p50': 1.0}
[PASS] C :: the national SMR capacity reproduces the mandate trajectory in the fbB runs -- tiers: {'fbB_eia_p50': 'reprodu

In [20]:
# ---- phase D: cross-case invariance ----------------------------------------------------------
lif = load_invariance([c for c in CASES if c != "fbR_none"], {c: "fbR_none" for c in CASES})
lif.to_csv(SF / "load_invariance.csv", index=False)
record("D", "the exogenous load is identical in all 19 runs", "PASS" if lif.max_rel_diff.max() < 1e-6 else "FAIL",
       f"max rel difference {lif.max_rel_diff.max():.2e} vs fbR_none")
gate_data_scan(record, "D", CASES)
gate_national_totals(record, "D", CASES)
bi = input_identity(LEARNOFF_CASES, {c: BASE_OF_WORLD[META[c]["world"]] for c in LEARNOFF_CASES})
bi = bi.rename(columns={})
bi.to_csv(SF / "base_identity.csv", index=False)
record("D", "the learning-off outputs match the Step 3 base-run outputs (11 cases, both keys)",
       "PASS" if bi.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {bi.max_rel_diff.max():.2e}")
gate_dual_conversion(record, "D", CASES, DUAL, RAW)
pre = {c: float(load(c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES}
record("D", "no SMR capacity exists before 2031", "PASS" if max(pre.values()) == 0 else "FAIL",
       f"max pre-2031 SMR capacity {max(pre.values()):.1f} MW")
record("D", "where the runs build large reactors (context)", "INFO",
       {c: round(float(LARGE_NEW[c].sum()) / 1000.0, 2) for c in CASES if float(LARGE_NEW[c].sum()) > 0}
       or "no new large builds after 2030 in any of the 19 runs")

# ---- the shadow-price table of the 19 runs (cap_MW = national SMR capacity) --------------------
dualsf, *_ = duals_table(CASES, YEARS_RUN, {c: (TRAJ_W[META[c]["world"]] if META[c]["mandate"] else None) for c in CASES},
                         SMR_CAP, DUAL, RAW)
dualsf.to_csv(SF / "duals_by_year.csv", index=False)
assert set(dualsf["case"]) == set(CASES) and len(dualsf) == 19 * 16
summaryf = write_registry("itcfb_checks")

[PASS] D :: the exogenous load is identical in all 19 runs -- max rel difference 0.00e+00 vs fbR_none
[PASS] D :: no data value is NaN or infinite in any file -- all 209 keys x 19 files scanned (key_inventory.n_nonfinite)
[PASS] D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (key_inventory.min_*)
[PASS] D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 19 cases (storage charging and upgrades make some non-nuclear rows negative; a normal convention)


[PASS] D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 19 cases within 1e-3


[PASS] D :: the learning-off outputs match the Step 3 base-run outputs (11 cases, both keys) -- max rel difference 0.00e+00
[PASS] D :: the dual conversion is correct (raw = cost_scale x pvf_onm x converted) -- max rel err 6.46e-08 across all cases with duals
[PASS] D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW
[INFO] D :: where the runs build large reactors (context) -- no new large builds after 2030 in any of the 19 runs
itcfb_checks {'PASS': 38, 'INFO': 10}


## 5. The itcfbm batch (`itcfbm_checks`): 24 below-headline credit runs

The minus runs feed the headline required ITC rate minus a decrement d (rungs m01, m05, m10, m15:
1, 5, 10, 15 rate points), with no bump, in three worlds (aj, mck, eo) and two arms: **fbBm**
(learning off) and **fbCm** (learning on). No run has a mandate. Under-delivery is the measured
object, so every phase-C row is INFO. On top of the delivery tiers, an **outcome classifier** maps
each run to the pre-registered reading: *full-delivery* (tier exact or reproduces),
*delayed-start-recover* (a rate-year shortfall, but the 2050 shortfall inside the reproduces gate),
*partial* (2050 shortfall within 15 %), *under-delivery* (above 15 % with builds), *no-build-lock-in*
(national SMR capacity at or below 5 MW in every year).

Derived tables and readers (notebook 09): `trajectory_gaps.csv`, `delivery_outcomes.csv`,
`incentives_echo.csv` (the fed rates), `checks_summary.csv`. The minus runs have no shadow price, so
there is no `duals_by_year.csv`.

In [21]:
record = registry("itcfbm_checks")
SM = out_dir("itcfbm_checks")

cases_m = pd.read_csv(paths.casefile("itcfbm"), index_col=0)
CASES = [c for c in cases_m.columns if c != "Default Value"]
assert len(CASES) == 24, f"the itcfbm casefile has 24 runs, got {len(CASES)}"
sw = lambda c, n: switch(cases_m, c, n)
WORLDS = ["aj", "mck", "eo"]
DECS = {"m01": 0.01, "m05": 0.05, "m10": 0.10, "m15": 0.15}
RUNGS = list(DECS)

def parse_case(c):
    tok, world = c.split("_")[0], c.split("_")[1]
    arm, rung = tok[:-2], "m" + tok[-2:]
    assert arm in ("fbBm", "fbCm") and rung in DECS and world in WORLDS, c
    return arm, rung, world

META = {}
for c in CASES:
    arm, rung, world = parse_case(c)
    META[c] = dict(arm=arm, rung=rung, world=world, mandate=int(float(sw(c, "GSw_NuclearCapMandate"))),
                   learning=int(float(sw(c, "GSw_NuclearLearning"))), suffix=sw(c, "incentives_suffix"),
                   foreign=sw(c, "GSw_NuclearLearning_ForeignScen"),
                   pc={"nuclear": sw(c, "plantchar_nuclear"), "nuclear-smr": sw(c, "plantchar_nuclear_smr")},
                   fin=sw(c, "financials_tech_suffix"))
ARM_CASES = {a: [c for c in CASES if META[c]["arm"] == a] for a in ["fbBm", "fbCm"]}
assert [len(ARM_CASES[a]) for a in ["fbBm", "fbCm"]] == [12, 12]
case_of = lambda arm, rung, world: f"{arm}{rung[1:]}_{world}_p50"
BASE_OF_WORLD = {w: f"smr100_{w}_p50" for w in WORLDS}
BASE_CASES = sorted(BASE_OF_WORLD.values())
SIBLING = {("fbBm", w): f"fbB_{w}_p50" for w in WORLDS}
SIBLING.update({("fbCm", w): f"fbC_{w}_p50" for w in WORLDS})
SIBLING_CASES = sorted(set(SIBLING.values()))

# ---- fed rates from the incentives files; rate years = the main set's rate years -------------
HEAD_YEARS = {w: sorted(RATES[w]) for w in WORLDS}          # RATES: the itcfb main-set files (section 4)
RATES_M, M_EXP_M = {}, {}
for w in WORLDS:
    for g in RUNGS:
        inc = pd.read_csv(FIN_DIR / f"incentives_obbba_itcfb{g}_{w}_p50.csv")
        n = inc[inc["i"] == "Nuclear-SMR"]
        RATES_M[(w, g)] = {int(t): float(f) for t, f in zip(n["t_start_construction"], n["itc_frac"])}
        assert sorted(RATES_M[(w, g)]) == HEAD_YEARS[w], f"{w}/{g}: the rung file credits the main set's rate years"
        M_EXP_M[(w, g)] = {t: f * (1.0 - PEN) for t, f in RATES_M[(w, g)].items()}

# ---- phase A ----------------------------------------------------------------------------------
gate_census(record, "A", "itcfbm", CASES, "one output file exists for each of the 24 cases")
missing_ref = [c for c in BASE_CASES + SIBLING_CASES + [REF4]
               if not INV.set_index(["batch", "case"]).loc[(RUN_BATCH[c], c), "returned"]]
record("A", "the reference files are available for comparison (3 base cases, 6 siblings, 1 key-vintage reference)",
       "PASS" if not missing_ref else "FAIL", missing_ref or "all present (key_inventory)", port="adapted")
record("A", "the folder census matches the consolidated delivery", "INFO",
       f"{int((INV.batch == 'itcfbm').sum())} itcfbm runs in the key inventory", port="adapted")
keysets, union = gate_keys(record, "A", CASES, "all 24 files have the same data keys", ref=REF4,
                           ref_check="the key set agrees with the Step 4 vintage (separate-delivery drift detector)")
EXPECT_YEARS = [int(y) for y in sw(CASES[0], "yearset").split("_")]
SOLVE_YEARS = [t for t in EXPECT_YEARS if t >= 2026]
YEARS_RUN = EXPECT_YEARS
obj = gate_solve_health(record, "A", CASES, EXPECT_YEARS)

[PASS] A :: one output file exists for each of the 24 cases -- 24 runs returned (key_inventory: batch itcfbm)
[PASS] A :: the reference files are available for comparison (3 base cases, 6 siblings, 1 key-vintage reference) -- all present (key_inventory)
[INFO] A :: the folder census matches the consolidated delivery -- 24 itcfbm runs in the key inventory
[PASS] A :: all 24 files have the same data keys -- key counts [207]; runs without a shadow-price table: 24
[PASS] A :: the key set agrees with the Step 4 vintage (separate-delivery drift detector) -- 209 keys, no drift
[PASS] A :: the solver residuals are small in every file -- worst |residual| 7.37e-04 at fbCm05_mck_p50
[PASS] A :: the objective value is a normal positive number in every file -- range 1.4026e+12 to 1.6371e+12
[PASS] A :: the model years are correct in every file (annual 2031-2035 block present) -- expected [2010, 2015, 2020, 2023, 2026, 2029, 2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]


[PASS] A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 at 14.7645 (max dev 9.5e-07); cost_scale==1; z_rep has 16 years in every file


In [22]:
# ---- B1: design matrix ------------------------------------------------------------------------
mx_bad = []
for c in CASES:
    m_ = META[c]
    ok = (m_["mandate"] == 0 and m_["learning"] == {"fbBm": 0, "fbCm": 1}[m_["arm"]]
          and m_["suffix"] == f"obbba_itcfbm{m_['rung'][1:]}_{m_['world']}_p50")
    if m_["learning"]:
        ok = ok and m_["foreign"] == f"fb_{m_['world']}_p50"
    if not ok:
        mx_bad.append(c)
record("B", "the case matrix encodes the intended arm, rung, world, and learning design",
       "PASS" if not mx_bad else "FAIL", mx_bad or "24 cases match the two-arm four-rung design (mandate 0 everywhere)")

# ---- B2: incentives files (conventions; the t09 equality moves to notebook 09) -----------------
echo_rows, b2_bad = [], []
for w in WORLDS:
    for g in RUNGS:
        inc = pd.read_csv(FIN_DIR / f"incentives_obbba_itcfb{g}_{w}_p50.csv")
        nuc_rows = inc[inc["i"].str.lower().str.startswith("nuclear")]
        only_smr = set(nuc_rows["i"]) == {"Nuclear-SMR"}
        conv_ok = bool((nuc_rows["safe_harbor"] == 0).all()
                       and (nuc_rows["t_max_online"] == nuc_rows["t_start_construction"]).all()
                       and (nuc_rows["itc_tax_equity_penalty"] == PEN).all()
                       and (nuc_rows["itc_energy_comm_bonus"] == 0.0).all()
                       and (nuc_rows["itc_percpt_domestic_bonus"] == 0.0).all())
        if not (only_smr and conv_ok):
            b2_bad.append((w, g, dict(only_smr=only_smr, conv_ok=conv_ok)))
        for t, f in sorted(RATES_M[(w, g)].items()):
            echo_rows.append(dict(world=w, rung=g, t=t, itc_frac_file=f))
pd.DataFrame(echo_rows).to_csv(SM / "incentives_echo.csv", index=False)
record("B", "the fed incentives files follow the credit conventions at the rate years", "PASS" if not b2_bad else "FAIL",
       b2_bad or "12 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0", port="adapted")
record("B", "the fed incentives files equal the headline rate schedule minus the rung decrement at exactly the rate years",
       "INFO", "moved to notebook 09: it needs t09_required_itc.csv from notebook 07 (itc_frac == "
       "round(i_model_headline - d, 3))", port="moved")

# ---- B2b: cross-set offset: at rung d the minus rate sits 0.01 + d below the main-set rate -------
off_rows, b2b_bad = [], []
for w in WORLDS:
    for g, d in DECS.items():
        for t, f in RATES_M[(w, g)].items():
            got_off = RATES[w][t] - f
            ok = abs(got_off - (EPS + d)) < 5e-4
            off_rows.append(dict(world=w, rung=g, t=t, main_itc_frac=RATES[w][t], minus_itc_frac=f,
                                 offset=round(got_off, 4), offset_expected=EPS + d, ok=ok))
            if not ok:
                b2b_bad.append((w, g, t, round(got_off, 4)))
pd.DataFrame(off_rows).to_csv(SM / "sibling_rate_offset.csv", index=False)
record("B", "the minus rates sit exactly 0.01 + d below the main-set sibling rates at every rate year",
       "PASS" if not b2b_bad else "FAIL",
       b2b_bad[:5] or f"{len(off_rows)} (world, rung, year) points; offset == bump + decrement everywhere")

# ---- rate recovery ----------------------------------------------------------------------------
rr_rows = []
for c in CASES:
    g_ = recovered_m(c, META[c]["fin"]).groupby(["i", "t"])["m_rec"].agg(["median", "min", "max"])
    g_["spread_r"] = g_["max"] - g_["min"]
    w, rung = META[c]["world"], META[c]["rung"]
    for (i, t), r in g_.iterrows():
        m_exp = M_EXP_M[(w, rung)].get(t, 0.0) if i == "nuclear-smr" else 0.0
        rr_rows.append(dict(case=c, arm=META[c]["arm"], rung=rung, i=i, t=t, m_expected=round(m_exp, 4),
                            m_recovered=round(float(r["median"]), 4), spread_r=round(float(r["spread_r"]), 5),
                            delta=round(float(r["median"]) - m_exp, 4)))
rrm = pd.DataFrame(rr_rows)
rrm.to_csv(SM / "rate_recovery.csv", index=False)

def rate_year_rows_m(case_list):
    sub = rrm[rrm["case"].isin(case_list) & (rrm["i"] == "nuclear-smr")]
    return sub[[t in HEAD_YEARS[META[c]["world"]] for c, t in zip(sub["case"], sub["t"])]]

b4 = rate_year_rows_m(ARM_CASES["fbBm"])
w4 = b4.loc[b4["delta"].abs().idxmax()]
record("B", "the recovered ITC rate equals the fed rate at exactly the rate years (fbBm)",
       "PASS" if b4["delta"].abs().max() <= TOL_M else "FAIL",
       f"{len(b4)} rate case-years; worst |dm| {b4['delta'].abs().max():.4f} at ({w4.case}, {w4.t})")
b4c = rate_year_rows_m(ARM_CASES["fbCm"])
w4c = b4c.loc[b4c["delta"].abs().idxmax()]
record("B", "the recovered ITC rate equals the fed rate in the learning runs (fbCm)",
       "PASS" if b4c["delta"].abs().max() <= TOL_M else "INFO",
       f"{len(b4c)} rate case-years; worst |dm| {b4c['delta'].abs().max():.4f} at ({w4c.case}, {w4c.t})")
off = rrm[[not (i == "nuclear-smr" and t in HEAD_YEARS[META[c]["world"]]) for c, i, t in zip(rrm["case"], rrm["i"], rrm["t"])]]
record("B", "the recovered rate is zero outside the rate years and for the large reactor in all 24 runs",
       "PASS" if off["m_recovered"].abs().max() <= TOL_M else "FAIL",
       f"{len(off)} points; max |m| {off['m_recovered'].abs().max():.4f}")
sp = rrm[rrm["i"] == "nuclear-smr"]
record("B", "the recovered rate is uniform across regions", "PASS" if sp["spread_r"].max() <= TOL_M else "FAIL",
       f"max spread over regions {sp['spread_r'].max():.5f} (24 fed cases)")
late = rrm[(rrm["t"] >= 2036) & (rrm["m_expected"] > 0)]
record("B", "the phaseout does not reduce the nuclear credit in any year (exemption held)",
       "PASS" if late["delta"].abs().max() <= TOL_M else "FAIL",
       f"{len(late)} credited case-years at t >= 2036; worst |dm| {late['delta'].abs().max():.4f}")
leak_rows = itc_wedge_leak([("fbBm05_aj_p50", "smr100_aj_p50"), ("fbCm15_eo_p50", "smr100_eo_p50"),
                            ("fbCm01_mck_p50", "smr100_mck_p50")])
record("B", "the phaseout exemption did not change the other technologies (non-nuclear ITC wedge identical to the base run)",
       "PASS" if not leak_rows else "FAIL", leak_rows or "3 spot cases vs their base runs; wedge identical")
# The model pays the credit only on builds: builds without payments is a defect; no builds and no
# payments is the lock-in signature (phase C).
SMR_NEW = {c: float(load(c, "cap_new_ann").query("i == 'nuclear-smr' and t > 2030")["Value"].sum()) for c in CASES}
pay_bad, no_build = [], []
for c in CASES:
    sc = load(c, "systemcost_techba")
    v = float(sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]["Value"].abs().sum())
    if SMR_NEW[c] > 100.0 and v < 1e6:
        pay_bad.append((c, f"builds {SMR_NEW[c]:,.0f} MW but no nuclear ITC payments"))
    if SMR_NEW[c] <= 100.0:
        no_build.append(c)
record("B", "the system cost data shows nuclear ITC payments wherever SMR builds occurred",
       "PASS" if not pay_bad else "FAIL",
       pay_bad or f"payments present in every case with builds; cases with ~no builds: {no_build or 'none'}")
itc_mag = {c: abs(float(load(c, "tax_expenditure_itc")["Value"].sum())) for c in CASES}
record("B", "the other technologies keep their ITC in all 24 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
       f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")
fpm = cost_fingerprint(CASES, {c: META[c]["pc"] for c in CASES}, {c: META[c]["learning"] for c in CASES})
fpm.to_csv(SM / "fingerprint_errors.csv", index=False)
off_fp = fpm[fpm["learning"] == 0]
w = off_fp.loc[off_fp.rel_err.idxmax()]
record("B", "the learning-off runs use the drawn cost path without change (12 fbBm)",
       "PASS" if off_fp.rel_err.max() < 1e-4 else "FAIL",
       f"{len(off_fp)} (case, tech, year) points; max rel err {off_fp.rel_err.max():.2e} at ({w.case}, {w.i}, {w.t})")
record("B", "the learning-on runs write an in-run cost path (deviation from the drawn path is expected; ~0 with zero builds is also expected)",
       "INFO", fpm[(fpm["learning"] == 1) & (fpm["t"] >= 2031)].groupby(["case", "i"])["rel_err"].max().round(4).to_dict())
b15_bad = sorted(c for c in CASES if KEYS[c] & DUAL_KEYS)
record("B", "no dual key exists in any of the 24 files (no mandate anywhere)", "PASS" if not b15_bad else "FAIL",
       b15_bad or "nuclear_cap_price absent from all 24 files")
b16_worst = {}
for w in WORLDS:
    group = [SIBLING[("fbBm", w)]] + [case_of("fbBm", g, w) for g in RUNGS]
    for key in ["cost_cap", "cost_cap_fin_mult_noITC"]:
        a0 = nuc_slice(group[0], key)
        for c in group[1:]:
            b16_worst[(w, key)] = max(b16_worst.get((w, key), 0.0), float((nuc_slice(c, key) - a0).abs().max()))
record("B", "the no-ITC finance multiplier and the capital cost are identical across each world's learning-off ladder (sibling + four rungs)",
       "PASS" if max(b16_worst.values()) < 1e-6 else "FAIL",
       f"max |difference| {max(b16_worst.values()):.2e} over {sorted(set(k[0] for k in b16_worst))}")

[PASS] B :: the case matrix encodes the intended arm, rung, world, and learning design -- 24 cases match the two-arm four-rung design (mandate 0 everywhere)
[PASS] B :: the fed incentives files follow the credit conventions at the rate years -- 12 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0
[INFO] B :: the fed incentives files equal the headline rate schedule minus the rung decrement at exactly the rate years -- moved to notebook 09: it needs t09_required_itc.csv from notebook 07 (itc_frac == round(i_model_headline - d, 3))
[PASS] B :: the minus rates sit exactly 0.01 + d below the main-set sibling rates at every rate year -- 120 (world, rung, year) points; offset == bump + decrement everywhere


[PASS] B :: the recovered ITC rate equals the fed rate at exactly the rate years (fbBm) -- 120 rate case-years; worst |dm| 0.0001 at (fbBm01_aj_p50, 2038)
[PASS] B :: the recovered ITC rate equals the fed rate in the learning runs (fbCm) -- 120 rate case-years; worst |dm| 0.0001 at (fbCm01_mck_p50, 2032)
[PASS] B :: the recovered rate is zero outside the rate years and for the large reactor in all 24 runs -- 336 points; max |m| 0.0000
[PASS] B :: the recovered rate is uniform across regions -- max spread over regions 0.00105 (24 fed cases)
[PASS] B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 120 credited case-years at t >= 2036; worst |dm| 0.0001


[PASS] B :: the phaseout exemption did not change the other technologies (non-nuclear ITC wedge identical to the base run) -- 3 spot cases vs their base runs; wedge identical


[PASS] B :: the system cost data shows nuclear ITC payments wherever SMR builds occurred -- payments present in every case with builds; cases with ~no builds: ['fbCm15_aj_p50']
[PASS] B :: the other technologies keep their ITC in all 24 runs -- total ITC tax expenditure magnitude 125.3 to 1136.6 B$ per case


[PASS] B :: the learning-off runs use the drawn cost path without change (12 fbBm) -- 984 (case, tech, year) points; max rel err 1.82e-07 at (fbBm01_eo_p50, nuclear-smr, 2050)
[INFO] B :: the learning-on runs write an in-run cost path (deviation from the drawn path is expected; ~0 with zero builds is also expected) -- {('fbCm01_aj_p50', 'nuclear'): 0.0, ('fbCm01_aj_p50', 'nuclear-smr'): 0.0394, ('fbCm01_eo_p50', 'nuclear'): 0.0303, ('fbCm01_eo_p50', 'nuclear-smr'): 0.0591, ('fbCm01_mck_p50', 'nuclear'): 0.0034, ('fbCm01_mck_p50', 'nuclear-smr'): 0.1407, ('fbCm05_aj_p50', 'nuclear'): 0.0, ('fbCm05_aj_p50', 'nuclear-smr'): 0.0436, ('fbCm05_eo_p50', 'nuclear'): 0.0387, ('fbCm05_eo_p50', 'nuclear-smr'): 0.0722, ('fbCm05_mck_p50', 'nuclear'): 0.0031, ('fbCm05_mck_p50', 'nuclear-smr'): 0.2406, ('fbCm10_aj_p50', 'nuclear'): 0.0, ('fbCm10_aj_p50', 'nuclear-smr'): 0.1493, ('fbCm10_eo_p50', 'nuclear'): 0.1041, ('fbCm10_eo_p50', 'nuclear-smr'): 0.2102, ('fbCm10_mck_p50', 'nuclear'): 0.0043, ('fbC

In [23]:
# ---- phase C: delivery (INFO by design) --------------------------------------------------------
SMR_CAP = {c: load(c, "cap").query("i == 'nuclear-smr'").groupby("t")["Value"].sum() for c in CASES}
LARGE_NEW = {c: load(c, "cap_new_ann").query("i == 'nuclear' and t > 2030").groupby("t")["Value"].sum() for c in CASES}
gap_rows = []
for c in CASES:
    w = META[c]["world"]
    for t in YEARS_RUN:
        req = float(TRAJ_W[w].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP[c].get(t, 0.0))
        gap_rows.append(dict(case=c, arm=META[c]["arm"], rung=META[c]["rung"], world=w, t=t,
                             status="rate" if t in HEAD_YEARS[w] else "slack", mandate_MW=req,
                             cap_MW=round(have, 1), gap_MW=round(have - req, 1),
                             gap_pct=round(100 * (have - req) / req, 2)))
gapsm = pd.DataFrame(gap_rows)
gapsm.to_csv(SM / "trajectory_gaps.csv", index=False)
TIER_M = {c: gap_tier(gapsm, c) for c in CASES}
for arm in ["fbBm", "fbCm"]:
    record("C", f"the delivery tiers of the {arm} runs (a finding, not a pass condition)", "INFO",
           f"tiers: { {c: TIER_M[c] for c in ARM_CASES[arm]} }")

out_rows = []
for c in CASES:
    w = META[c]["world"]
    sub = gapsm[gapsm["case"] == c]
    end = sub[sub["t"] == 2050]
    end_short, end_req = float(-end["gap_MW"].iloc[0]), float(end["mandate_MW"].iloc[0])
    cap_max = float(SMR_CAP[c].max()) if len(SMR_CAP[c]) else 0.0
    gate = sub[sub["status"] == "rate"]
    tol1 = np.maximum(0.05 * gate["mandate_MW"], 500.0)
    rate_miss = bool(((-gate["gap_MW"]).clip(lower=0.0) > tol1).any())
    end_ok = end_short <= max(0.05 * end_req, 500.0)
    if cap_max <= 5.0:
        outcome = "no-build-lock-in"
    elif TIER_M[c] in ("exact", "reproduces"):
        outcome = "full-delivery"
    elif rate_miss and end_ok:
        outcome = "delayed-start-recover"
    elif end_short <= 0.15 * end_req:
        outcome = "partial"
    else:
        outcome = "under-delivery"
    built = SMR_CAP[c][SMR_CAP[c] > 5.0]
    out_rows.append(dict(case=c, arm=META[c]["arm"], rung=META[c]["rung"], world=w, tier=TIER_M[c], outcome=outcome,
                         first_build_year=int(built.index.min()) if len(built) else 0,
                         smr_2050_MW=round(float(SMR_CAP[c].get(2050, 0.0)), 1), traj_2050_MW=end_req,
                         delivered_frac_2050=round(float(SMR_CAP[c].get(2050, 0.0)) / end_req, 4)))
outcomes = pd.DataFrame(out_rows)
outcomes.to_csv(SM / "delivery_outcomes.csv", index=False)
record("C", "the pre-registered outcome classification of all 24 runs", "INFO",
       outcomes.groupby(["arm", "outcome"])["case"].count().to_dict())
display(outcomes)

SIB_CAP = {s: load(s, "cap").query("i == 'nuclear-smr'").groupby("t")["Value"].sum() for s in SIBLING_CASES}
mono_notes = []
for arm in ["fbBm", "fbCm"]:
    for w in WORLDS:
        ladder = [(g, float(SMR_CAP[case_of(arm, g, w)].get(2050, 0.0))) for g in ["m15", "m10", "m05", "m01"]]
        ladder.append(("sibling+0.01", float(SIB_CAP[SIBLING[(arm, w)]].get(2050, 0.0))))
        viol = [(a[0], b[0]) for a, b in zip(ladder, ladder[1:]) if a[1] > b[1] + 5.0]
        if viol:
            mono_notes.append(f"{arm}/{w}: inversions {viol}")
record("C", "the 2050 delivery increases weakly up the rate ladder within each (arm, world)", "INFO",
       mono_notes or "no inversion: m15 <= m10 <= m05 <= m01 <= sibling in all 6 ladders")
record("C", "the large-reactor substitution in the minus runs (a report, not a failure)", "INFO",
       {c: round(float(LARGE_NEW[c].sum()) / 1000.0, 2) for c in CASES if float(LARGE_NEW[c].sum()) > 0}
       or "no new large builds after 2030 in any of the 24 runs")

# ---- phase D ----------------------------------------------------------------------------------
lim = load_invariance(CASES, {c: "fbB_aj_p50" for c in CASES})
lim.to_csv(SM / "load_invariance.csv", index=False)
record("D", "the exogenous load is identical in all 24 runs and equal to the main-set sibling's load",
       "PASS" if lim.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {lim.max_rel_diff.max():.2e} vs fbB_aj_p50")
gate_data_scan(record, "D", CASES)
gate_national_totals(record, "D", CASES)
bim = input_identity(ARM_CASES["fbBm"], {c: BASE_OF_WORLD[META[c]["world"]] for c in ARM_CASES["fbBm"]})
bim.to_csv(SM / "base_identity.csv", index=False)
record("D", "the fbBm nuclear input parameters match the Step 3 base-run parameters (12 cases, both keys)",
       "PASS" if bim.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {bim.max_rel_diff.max():.2e}")
pre = {c: float(load(c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES}
record("D", "no SMR capacity exists before 2031", "PASS" if max(pre.values()) == 0 else "FAIL",
       f"max pre-2031 SMR capacity {max(pre.values()):.1f} MW")
record("D", "where the runs build large reactors (context)", "INFO",
       {c: round(float(LARGE_NEW[c].sum()) / 1000.0, 2) for c in CASES if float(LARGE_NEW[c].sum()) > 0}
       or "no new large builds after 2030 in any of the 24 runs")
summarym = write_registry("itcfbm_checks")

[INFO] C :: the delivery tiers of the fbBm runs (a finding, not a pass condition) -- tiers: {'fbBm01_aj_p50': 'reproduces', 'fbBm01_mck_p50': 'reproduces', 'fbBm01_eo_p50': 'reproduces', 'fbBm05_aj_p50': 'partial', 'fbBm05_mck_p50': 'partial', 'fbBm05_eo_p50': 'partial', 'fbBm10_aj_p50': 'no', 'fbBm10_mck_p50': 'no', 'fbBm10_eo_p50': 'no', 'fbBm15_aj_p50': 'no', 'fbBm15_mck_p50': 'no', 'fbBm15_eo_p50': 'no'}
[INFO] C :: the delivery tiers of the fbCm runs (a finding, not a pass condition) -- tiers: {'fbCm01_aj_p50': 'reproduces', 'fbCm01_mck_p50': 'reproduces', 'fbCm01_eo_p50': 'reproduces', 'fbCm05_aj_p50': 'no', 'fbCm05_mck_p50': 'no', 'fbCm05_eo_p50': 'partial', 'fbCm10_aj_p50': 'no', 'fbCm10_mck_p50': 'no', 'fbCm10_eo_p50': 'no', 'fbCm15_aj_p50': 'no', 'fbCm15_mck_p50': 'no', 'fbCm15_eo_p50': 'no'}
[INFO] C :: the pre-registered outcome classification of all 24 runs -- {('fbBm', 'delayed-start-recover'): 1, ('fbBm', 'full-delivery'): 3, ('fbBm', 'partial'): 2, ('fbBm', 'under-deliv

,case,arm,rung,world,tier,outcome,first_build_year,smr_2050_MW,traj_2050_MW,delivered_frac_2050
0,fbBm01_aj_p50,fbBm,m01,aj,reproduces,full-delivery,2031,57004.8,36399.0,1.5661
1,fbBm01_mck_p50,fbBm,m01,mck,reproduces,full-delivery,2031,139191.1,103399.0,1.3462
2,fbBm01_eo_p50,fbBm,m01,eo,reproduces,full-delivery,2031,348236.6,303399.0,1.1478
3,fbBm05_aj_p50,fbBm,m05,aj,partial,partial,2031,33388.2,36399.0,0.9173
4,fbBm05_mck_p50,fbBm,m05,mck,partial,partial,2031,88117.7,103399.0,0.8522
5,fbBm05_eo_p50,fbBm,m05,eo,partial,delayed-start-recover,2031,289517.6,303399.0,0.9542
6,fbBm10_aj_p50,fbBm,m10,aj,no,under-delivery,2031,21957.4,36399.0,0.6032
7,fbBm10_mck_p50,fbBm,m10,mck,no,under-delivery,2031,47995.6,103399.0,0.4642
8,fbBm10_eo_p50,fbBm,m10,eo,no,under-delivery,2031,221441.5,303399.0,0.7299
9,fbBm15_aj_p50,fbBm,m15,aj,no,under-delivery,2031,12964.4,36399.0,0.3562


[INFO] C :: the 2050 delivery increases weakly up the rate ladder within each (arm, world) -- no inversion: m15 <= m10 <= m05 <= m01 <= sibling in all 6 ladders
[INFO] C :: the large-reactor substitution in the minus runs (a report, not a failure) -- no new large builds after 2030 in any of the 24 runs


[PASS] D :: the exogenous load is identical in all 24 runs and equal to the main-set sibling's load -- max rel difference 0.00e+00 vs fbB_aj_p50
[PASS] D :: no data value is NaN or infinite in any file -- all 207 keys x 24 files scanned (key_inventory.n_nonfinite)
[PASS] D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (key_inventory.min_*)
[PASS] D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 24 cases (storage charging and upgrades make some non-nuclear rows negative; a normal convention)


[PASS] D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 24 cases within 1e-3


[PASS] D :: the fbBm nuclear input parameters match the Step 3 base-run parameters (12 cases, both keys) -- max rel difference 0.00e+00
[PASS] D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW
[INFO] D :: where the runs build large reactors (context) -- no new large builds after 2030 in any of the 24 runs
itcfbm_checks {'PASS': 29, 'INFO': 9}


## 6. Run sanity summary

The table counts the registry outcomes of the four batches. The QA gate that notebooks 07 to 09
apply is: no FAIL in phases A, B (B1 to B3 for step4) and D. A FAIL in phase C of itcfb would be a
pre-registered finding, not a defect. The source registries counted 44 PASS / 10 INFO (step3),
44 PASS / 8 INFO (step4), 38 PASS / 9 INFO (itcfb) and 29 PASS / 8 INFO (itcfbm); the port keeps
every gate that the extracts can run, so the counts here differ only by the `adapted` and `moved`
rows listed in section 1.

In [24]:
rows = []
for name, s in [("step3_checks", summary3), ("step4_checks", summary4), ("itcfb_checks", summaryf),
                ("itcfbm_checks", summarym)]:
    cnt = s["status"].value_counts()
    rows.append(dict(registry=name, n=len(s), PASS=int(cnt.get("PASS", 0)), FAIL=int(cnt.get("FAIL", 0)),
                     INFO=int(cnt.get("INFO", 0)), adapted=int((s["port"] == "adapted").sum()),
                     moved=int((s["port"] == "moved").sum())))
    hard = s[(s["status"] == "FAIL") & ~s["phase"].isin(["C"])]
    assert not len(hard), f"mechanical-integrity FAIL in {name}:\n{hard}"
sanity = pd.DataFrame(rows)
display(sanity)
print("files written:")
for p in sorted(OUT.rglob("*.csv")):
    print(" ", paths.rel(p), f"{p.stat().st_size / 1024:.1f} kB")

,registry,n,PASS,FAIL,INFO,adapted,moved
0,step3_checks,54,42,0,12,5,2
1,step4_checks,52,43,0,9,7,2
2,itcfb_checks,48,38,0,10,8,1
3,itcfbm_checks,38,29,0,9,8,1


files written:
  analysis\results\exports\06a\itcfb_checks\base_identity.csv 1.2 kB
  analysis\results\exports\06a\itcfb_checks\checks_summary.csv 8.4 kB
  analysis\results\exports\06a\itcfb_checks\duals_by_year.csv 14.1 kB
  analysis\results\exports\06a\itcfb_checks\fbA_collapse.csv 0.6 kB
  analysis\results\exports\06a\itcfb_checks\fbK_calibration.csv 17.1 kB
  analysis\results\exports\06a\itcfb_checks\fingerprint_errors.csv 121.3 kB
  analysis\results\exports\06a\itcfb_checks\incentives_echo.csv 0.9 kB
  analysis\results\exports\06a\itcfb_checks\load_invariance.csv 0.5 kB
  analysis\results\exports\06a\itcfb_checks\rate_recovery.csv 22.7 kB
  analysis\results\exports\06a\itcfb_checks\trajectory_gaps.csv 8.4 kB
  analysis\results\exports\06a\itcfbm_checks\base_identity.csv 1.4 kB
  analysis\results\exports\06a\itcfbm_checks\checks_summary.csv 7.1 kB
  analysis\results\exports\06a\itcfbm_checks\delivery_outcomes.csv 1.9 kB
  analysis\results\exports\06a\itcfbm_checks\fingerprint_error